## 1. Setup

In [ ]:
!pip install --upgrade pip setuptools wheel --quiet
!pip install numpy pandas matplotlib seaborn scipy sympy --quiet
!pip install torch torchvision --quiet
!pip install timm scikit-learn scikit-image --quiet
!pip install thop shap lime --quiet
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
!pip install -q -U huggingface_hub hf_xet

In [ ]:
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.functional import softmax
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split
from timm import create_model
from timm.layers import DropPath          
from thop import profile
from sklearn.manifold import TSNE
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, auc,
)

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)

if not os.path.exists("images"):
    os.makedirs("images")

QUICK_TEST_MODE = False

GLOBAL_NUM_EPOCHS = 10

GLOBAL_RANK = 32

GLOBAL_NUM_LAYERS = 2
GLOBAL_NUM_HEADS = 8
FORCE_RETRAIN = True

QUICK_TEST_MAX_TRAIN = 64
QUICK_TEST_MAX_VAL = 16
QUICK_TEST_MAX_TEST = 16
QUICK_TEST_BATCH_SIZE = 8

from torch.utils.data import Subset

def quick_subset(torch_dataset, max_n):
  
    if not QUICK_TEST_MODE:
        return torch_dataset
    n = min(max_n, len(torch_dataset))
    return Subset(torch_dataset, list(range(n)))

if QUICK_TEST_MODE:
    print("QUICK_TEST_MODE is ON - using tiny data subsets to smoke-test the notebook.")
    print("Set QUICK_TEST_MODE = False above before running for real results.\n")

In [ ]:
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

DATA_ROOT = "/kaggle/input" if IS_KAGGLE else os.environ.get("DATA_ROOT", "./data")

OUTPUT_ROOT = "/kaggle/working" if IS_KAGGLE else os.environ.get("OUTPUT_ROOT", "./outputs")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

_CRC7K_PATH = "Data_Set_Path"

DATASET_CONFIGS = {
    "Kather5k": {
        "path": "Data_Set_Path",
        "weight_decay": 5e-4,
        "restore_best_checkpoint": True,
        "use_class_balancing": True,
    },
    "CRC7k": {
        "path": _CRC7K_PATH,
        "weight_decay": 5e-4,
        "use_class_balancing": True,
    },
    "NCT100k": {
        "train_val_path": "Data_Set_Path",
        "test_path": _CRC7K_PATH,
        "weight_decay": 5e-4,
        "use_class_balancing": True,
    },
    "LC25000": {
        "train_val_path": "Data_Set_Path"
                           if IS_KAGGLE else f"{DATA_ROOT}/LC25000/Train and Validation Set",
        "test_path": "Data_Set_Path"
                     if IS_KAGGLE else f"{DATA_ROOT}/LC25000/Test Set",
        "weight_decay": 5e-4,
        "use_class_balancing": True,
    },
    "BreakHis": {
        "path": "Data_Set_Path",     
        "weight_decay": 5e-4,
        "use_class_balancing": True,
    },
}

# Fallbacks used only if a dataset config is missing the key above.
DEFAULT_WEIGHT_DECAY = 0.0
DEFAULT_RESTORE_BEST_CHECKPOINT = True
DEFAULT_USE_CLASS_BALANCING = False

def _check_dataset_paths(cfg):
    paths = [cfg["path"]] if "path" in cfg else [cfg["train_val_path"], cfg["test_path"]]
    return [p for p in paths if not os.path.isdir(p)]

SELECTED_DATASET = "NCT100k"   #Change the dataset key to run for the seleceted data or selecet #All to run the pipeline for all datasets

assert SELECTED_DATASET == "ALL" or SELECTED_DATASET in DATASET_CONFIGS, (
    f"Unknown SELECTED_DATASET '{SELECTED_DATASET}'. "
    f"Choose one of {list(DATASET_CONFIGS)} or 'ALL'."
)
DATASETS_TO_RUN = list(DATASET_CONFIGS.keys()) if SELECTED_DATASET == "ALL" else [SELECTED_DATASET]

print(f"Will run the pipeline for: {DATASETS_TO_RUN}")


In [ ]:
import urllib.request as _urlreq

GROUPS_CSV_OVERRIDE = None   
GROUPS_CSV_URL = "https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/main/kaggle/lc25000_image_groups.csv"
GROUPS_CSV_DOWNLOAD_TARGET = os.path.join(OUTPUT_ROOT, "lc25000_image_groups.csv")

def ensure_lc25000_groups_csv():
    if GROUPS_CSV_OVERRIDE and os.path.isfile(GROUPS_CSV_OVERRIDE):
        return GROUPS_CSV_OVERRIDE
    if os.path.isfile(GROUPS_CSV_DOWNLOAD_TARGET):
        return GROUPS_CSV_DOWNLOAD_TARGET
    print(f"Downloading LC25000-clean group mapping to {GROUPS_CSV_DOWNLOAD_TARGET} ...")
    try:
        _urlreq.urlretrieve(GROUPS_CSV_URL, GROUPS_CSV_DOWNLOAD_TARGET)
        print("Download succeeded.")
    except Exception as e:
        raise RuntimeError(
            f"Auto-download of the LC25000-clean group mapping failed ({e}). "
            f"Manual fallback: download {GROUPS_CSV_URL} on a machine with internet, "
            f"upload it (e.g. as a Kaggle Dataset via '+ Add Data'), and set "
            f"GROUPS_CSV_OVERRIDE above to its path."
        )
    return GROUPS_CSV_DOWNLOAD_TARGET


In [ ]:
%%writefile lc25000_dataset_utils.py
import torch
from PIL import Image


class PathListDataset(torch.utils.data.Dataset):
    
    def __init__(self, samples, transform=None):
        self.samples = samples  
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, label


class ClassesShim:
  
    def __init__(self, classes):
        self.classes = classes


In [ ]:
from lc25000_dataset_utils import PathListDataset, ClassesShim
from sklearn.model_selection import StratifiedGroupKFold as _SGKF_LC25000
from collections import Counter as _Counter_LC25000
import pandas as pd


def get_lc25000_pooled_samples(cfg):
    
    groups_csv = ensure_lc25000_groups_csv()
    groups_df = pd.read_csv(groups_csv)
    fname_to_group = dict(zip(groups_df["filename"], groups_df["group_id"]))

    train_val_path, test_path = cfg["train_val_path"], cfg["test_path"]
    class_names = sorted(d.name for d in os.scandir(train_val_path) if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    all_paths, all_groups, all_labels = [], [], []
    missing_from_map = 0
    for root in (train_val_path, test_path):
        for cname in class_names:
            class_dir = os.path.join(root, cname)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                if not fname.lower().endswith((".jpeg", ".jpg", ".png")):
                    continue
                gid = fname_to_group.get(fname)
                if gid is None:
                    missing_from_map += 1
                    continue
                all_paths.append(os.path.join(class_dir, fname))
                all_groups.append(gid)
                all_labels.append(class_to_idx[cname])

    print(f"[LC25000] Pooled {len(all_paths)} images "
          f"({missing_from_map} skipped - filename not in group mapping).")

    all_paths = np.array(all_paths, dtype=object)
    all_groups = np.array(all_groups)
    all_labels = np.array(all_labels)
    return all_paths, all_labels, all_groups, class_names


def build_lc25000_grouped_split(cfg, train_val_transform_, test_transform_, seed=42):

    all_paths, all_labels, all_groups, class_names = get_lc25000_pooled_samples(cfg)
    num_classes = len(class_names)
    print(f"[LC25000] Classes ({num_classes}): {class_names}")

    N_TEST_SPLITS = 6  
    trainval_idx, test_idx, _ = _split_from_balanced_groups(
        all_labels, all_groups, N_TEST_SPLITS, seed=seed, heldout_fold=seed % N_TEST_SPLITS
    )

    N_VAL_SPLITS = 10
    train_sub_idx, val_sub_idx, _ = _split_from_balanced_groups(
        all_labels[trainval_idx], all_groups[trainval_idx], N_VAL_SPLITS,
        seed=seed + 1000, heldout_fold=(seed + 1000) % N_VAL_SPLITS
    )
    train_idx = trainval_idx[train_sub_idx]
    val_idx = trainval_idx[val_sub_idx]

    g_train, g_val, g_test = set(all_groups[train_idx]), set(all_groups[val_idx]), set(all_groups[test_idx])
    assert not (g_train & g_val), "Group leakage between train and val!"
    assert not (g_train & g_test), "Group leakage between train and test!"
    assert not (g_val & g_test), "Group leakage between val and test!"

    print(f"[LC25000] Train: {len(train_idx):>6} imgs / {len(g_train):>4} groups")
    print(f"[LC25000] Val:   {len(val_idx):>6} imgs / {len(g_val):>4} groups")
    print(f"[LC25000] Test:  {len(test_idx):>6} imgs / {len(g_test):>4} groups")
    print("[LC25000] Group overlap across splits: NONE (verified).")
    print("[LC25000] Class balance (train):", _Counter_LC25000(all_labels[train_idx]))
    print("[LC25000] Class balance (val):  ", _Counter_LC25000(all_labels[val_idx]))
    print("[LC25000] Class balance (test): ", _Counter_LC25000(all_labels[test_idx]))

    train_dataset = PathListDataset(
        list(zip(all_paths[train_idx].tolist(), all_labels[train_idx].tolist())),
        transform=train_val_transform_,
    )
    val_dataset = PathListDataset(
        list(zip(all_paths[val_idx].tolist(), all_labels[val_idx].tolist())),
        transform=train_val_transform_,
    )
    test_dataset = PathListDataset(
        list(zip(all_paths[test_idx].tolist(), all_labels[test_idx].tolist())),
        transform=test_transform_,
    )
    classes_shim = ClassesShim(class_names)
    return train_dataset, val_dataset, test_dataset, num_classes, classes_shim


In [ ]:
import re
from sklearn.model_selection import GroupShuffleSplit

def try_extract_patient_id(filename):
    """Same patterns as the Section 22 audit - kept in sync with it."""
    patterns = [
        r'SOB_[A-Z]_[A-Z]+-(\d+-\d+)',   
        r'^(P\d+)',
        r'patient[_-]?(\d+)',
        r'case[_-]?(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, filename, flags=re.IGNORECASE)
        if m:
            return m.group(1)
    return None


def get_patient_groups(image_folder_dataset):
    
    filepaths = [s[0] for s in image_folder_dataset.samples]
    patient_ids = [try_extract_patient_id(os.path.basename(fp)) for fp in filepaths]
    if any(p is None for p in patient_ids) or len(set(patient_ids)) < 2:
        return None
    return np.array(patient_ids)


from sklearn.model_selection import StratifiedGroupKFold as _SGKF_BALANCED


def _balanced_group_folds(labels, groups, n_splits, seed, n_restarts=500):
    
    labels = np.asarray(labels)
    groups = np.asarray(groups)
    if labels.ndim != 1 or groups.ndim != 1 or len(labels) != len(groups):
        raise ValueError(f"labels/groups length mismatch: {len(labels)} vs {len(groups)}")
    unique_groups = np.unique(groups)
    if len(unique_groups) < n_splits:
        raise ValueError(f"Need at least {n_splits} groups, found {len(unique_groups)}")

    num_classes = int(labels.max()) + 1
    groups_per_class = [len(set(groups[labels == c])) for c in range(num_classes)]
    max_feasible_splits = min(groups_per_class) if groups_per_class else n_splits
    effective_n_splits = max(2, min(n_splits, max_feasible_splits))
    if effective_n_splits < n_splits:
        print(f"  [balanced_group_folds] Requested {n_splits} splits, but the scarcest class only "
              f"spans {max_feasible_splits} distinct groups - using {effective_n_splits} splits instead "
              f"(every fold still guaranteed to have all classes present).")
    n_splits = effective_n_splits  
    global_counts = np.bincount(labels, minlength=num_classes).astype(float)
    global_ratio = global_counts / max(global_counts.sum(), 1.0)
    target_size = len(labels) / float(n_splits)
    target_counts = global_counts / float(n_splits)

    best_assignment = None
    best_score = np.inf
    master = np.random.RandomState(seed)

    for _ in range(n_restarts):
        trial_seed = int(master.randint(0, 2**31 - 1))
        try:
            sgkf = _SGKF_BALANCED(n_splits=n_splits, shuffle=True, random_state=trial_seed)
            assignment = np.empty(len(labels), dtype=np.int64)
            for fold, (_, fold_idx) in enumerate(sgkf.split(np.zeros(len(labels)), labels, groups)):
                assignment[fold_idx] = fold
        except Exception:
            continue

        valid = True
        fold_counts = np.zeros((n_splits, num_classes), dtype=np.int64)
        fold_sizes = np.zeros(n_splits, dtype=np.int64)
        for f in range(n_splits):
            mask = assignment == f
            if not np.any(mask):
                valid = False
                break
            fold_counts[f] = np.bincount(labels[mask], minlength=num_classes)
            fold_sizes[f] = int(mask.sum())
            if np.any(fold_counts[f] == 0):
                valid = False
                break
        if not valid:
            continue

        ratios = fold_counts / np.maximum(fold_sizes[:, None], 1)
        ratio_error = np.mean(np.sum((ratios - global_ratio) ** 2, axis=1))
        count_error = np.mean(np.sum(((fold_counts - target_counts) / np.maximum(target_counts, 1.0)) ** 2, axis=1))
        size_error = np.mean(((fold_sizes - target_size) / target_size) ** 2)
        score = 8.0 * ratio_error + 1.0 * count_error + 0.25 * size_error

        if score < best_score:
            best_score = score
            best_assignment = assignment.copy()

    if best_assignment is None:
        raise RuntimeError(
            f"Could not construct {n_splits} valid group-level folds with all classes present. "
            f"Dataset has {len(unique_groups)} groups and class counts {global_counts.astype(int).tolist()}."
        )

    for f in range(n_splits):
        mask = best_assignment == f
        counts = np.bincount(labels[mask], minlength=num_classes)
        if np.any(counts == 0):
            raise RuntimeError(f"Fold {f+1} lacks a class: {counts.tolist()}")

    grouped_sets = [set(groups[best_assignment == f]) for f in range(n_splits)]
    for i in range(n_splits):
        for j in range(i + 1, n_splits):
            if grouped_sets[i] & grouped_sets[j]:
                raise RuntimeError(f"Group leakage between folds {i+1} and {j+1}")

    return best_assignment


def _split_from_balanced_groups(labels, groups, n_splits, seed, heldout_fold=0):
    fold_ids = _balanced_group_folds(labels, groups, n_splits, seed)
    idx = np.arange(len(labels))
    return idx[fold_ids != heldout_fold], idx[fold_ids == heldout_fold], fold_ids


def patient_group_split(image_folder_dataset, seed, test_frac=0.2, val_frac_of_trainval=0.1, verbose=False):
    
    groups = get_patient_groups(image_folder_dataset)
    if groups is None:
        return None

    idx = np.arange(len(image_folder_dataset))
    labels = np.asarray(image_folder_dataset.targets)
    if verbose:
        print(f"[patient_group_split seed={seed}] Images: {len(labels)}  Patients: {len(set(groups))}  "
              f"Classes: {np.bincount(labels, minlength=2)}")

    n_test = max(2, round(1 / test_frac))
    trainval_idx, test_idx, _ = _split_from_balanced_groups(labels, groups, n_test, seed, seed % n_test)

    n_val = max(2, round(1 / val_frac_of_trainval))
    tr_sub, val_sub, _ = _split_from_balanced_groups(
        labels[trainval_idx], groups[trainval_idx], n_val, seed + 1000, (seed + 1000) % n_val
    )
    train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]

    assert not (set(groups[train_idx]) & set(groups[val_idx]))
    assert not (set(groups[train_idx]) & set(groups[test_idx]))
    assert not (set(groups[val_idx]) & set(groups[test_idx]))

    if verbose:
        print(f"  Train: {len(train_idx)} images, {len(set(groups[train_idx]))} patients, "
              f"classes {np.bincount(labels[train_idx], minlength=2)}")
        print(f"  Val:   {len(val_idx)} images, {len(set(groups[val_idx]))} patients, "
              f"classes {np.bincount(labels[val_idx], minlength=2)}")
        print(f"  Test:  {len(test_idx)} images, {len(set(groups[test_idx]))} patients, "
              f"classes {np.bincount(labels[test_idx], minlength=2)}")

    return train_idx, val_idx, test_idx


In [ ]:
train_val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

batch_size = QUICK_TEST_BATCH_SIZE if QUICK_TEST_MODE else 32


In [ ]:
class ResNet18_Features(nn.Module):
    """Returns spatial feature map (B, 512, 7, 7) for a 224x224 input."""
    def __init__(self):
        super(ResNet18_Features, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])  

    def forward(self, x):
        return self.features(x)  


class DenseNet121_Features(nn.Module):
    """Returns spatial feature map (B, 1024, 7, 7) for a 224x224 input."""
    def __init__(self):
        super(DenseNet121_Features, self).__init__()
        densenet = models.densenet121(pretrained=True)
        self.features = densenet.features

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=False)  
        return x

In [ ]:
class LowRankSparseMultiheadAttention(nn.Module):
    
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention, self).__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        assert rank % num_heads == 0, "rank must be divisible by num_heads (rank-space multi-head split)"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.rank_head_dim = rank // num_heads  
        self.sparsity_ratio = sparsity_ratio

        self.q_low = nn.Linear(embed_dim, rank, bias=False)
        self.k_low = nn.Linear(embed_dim, rank, bias=False)
        self.v_low = nn.Linear(embed_dim, rank, bias=False)

        self.rank_mix = nn.Linear(rank, rank, bias=False)

        self.out_proj = nn.Linear(rank, embed_dim, bias=False)

        self.scale = self.rank_head_dim ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores
        num_to_keep = max(1, int(sparsity_ratio * seq_length))
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        return attn_scores.masked_fill(~sparse_mask, float('-inf'))

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()

        Q = self.q_low(x)  
        K = self.k_low(x)  
        V = self.v_low(x)  

        Q = Q.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)  
        K = K.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.rank_head_dim).transpose(1, 2)

        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  

        sparse_attn_scores = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)

        attn_output = torch.matmul(attn_probs, V)  

        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.rank)

        attn_output = self.rank_mix(attn_output)

        return self.out_proj(attn_output)

class CustomDeiTLayer(nn.Module):
    """Transformer encoder layer used by LoRaS-CT."""
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class HybridStudentModel(nn.Module):
    """LoRaS-CT student model (spatial tokens; grid_size controls N).
    grid_size=7 -> 49 tokens (native CNN output, no reduction)
    grid_size=3 -> 9 tokens (pooled down via adaptive_avg_pool2d)
    """
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, drop_path_rate=0.1, sparsity_ratio=0.5, grid_size=3):
        super(HybridStudentModel, self).__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024  

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_features=False):
        resnet_feats = self.resnet(x)      
        densenet_feats = self.densenet(x)  
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)  

        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))

        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)  

        x = self.deit_embed(combined_feats)  
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        pooled = x.mean(dim=1)  
        logits = self.classifier(pooled)
        if return_features:
            return logits, pooled
        return logits

In [ ]:
class CustomDeiTLayer_MHA(nn.Module):
   
    def __init__(self, embed_dim, num_heads, mlp_ratio=4., drop_path=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()  
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        normed = self.norm1(x)
        B, N, E = normed.shape
        Q = self.q_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(normed).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        attn_out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        attn_out = self.out_proj(attn_out)
        x = x + self.drop_path(attn_out)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class MHANetBaseline(nn.Module):
    """Same spatial CNN backbone as HybridStudentModel, but standard MHA (for fair comparison)."""
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 drop_path_rate=0.1, grid_size=3):
        super().__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        concat_channels = 512 + 1024
        self.grid_size = grid_size  

        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer_MHA(embed_dim, num_heads, drop_path=drop_path_rate)
            for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        resnet_feats = self.resnet(x)
        densenet_feats = self.densenet(x)
        combined_feats = torch.cat((resnet_feats, densenet_feats), dim=1)

        if self.grid_size != combined_feats.shape[-1]:
            combined_feats = F.adaptive_avg_pool2d(combined_feats, (self.grid_size, self.grid_size))

        b, c, h, w = combined_feats.shape
        combined_feats = combined_feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(combined_feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.classifier(x)
        return x

In [ ]:

class LinformerAttention(nn.Module):
    
    def __init__(self, embed_dim, num_heads, seq_len, proj_k=None):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.seq_len = seq_len
        self.proj_k = proj_k or max(1, seq_len // 2)  

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.E_proj = nn.Linear(seq_len, self.proj_k, bias=False)  
        self.F_proj = nn.Linear(seq_len, self.proj_k, bias=False)  
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        assert N == self.seq_len, f"LinformerAttention was built for N={self.seq_len}, got {N}"
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)          
        K = self.E_proj(self.k_proj(x).transpose(1, 2)).transpose(1, 2)                        
        V = self.F_proj(self.v_proj(x).transpose(1, 2)).transpose(1, 2)                        
        K = K.view(B, self.proj_k, self.num_heads, self.head_dim).transpose(1, 2)              
        V = V.view(B, self.proj_k, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)            
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class PerformerAttention(nn.Module):
   
    def __init__(self, embed_dim, num_heads, nb_features=None):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.nb_features = nb_features or self.head_dim

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.register_buffer(
            'random_matrix', torch.randn(self.num_heads, self.head_dim, self.nb_features)
        )

    def _phi(self, x):  
        proj = torch.einsum('bhnd,hdf->bhnf', x, self.random_matrix)
        return F.elu(proj) + 1

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        Qp, Kp = self._phi(Q), self._phi(K)
        KV = torch.einsum('bhnf,bhnd->bhfd', Kp, V)
        denom = torch.einsum('bhnf,bhf->bhn', Qp, Kp.sum(dim=2)) + 1e-6
        out = torch.einsum('bhnf,bhfd->bhnd', Qp, KV) / denom.unsqueeze(-1)
        out = out.transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class BigBirdAttention(nn.Module):
    
    def __init__(self, embed_dim, num_heads, block_size=64):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.block_size = block_size

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class StandardMHAWrapper(nn.Module):
    """Standard multi-head attention, written with explicit nn.Linear ops so
    thop can profile its FLOPs (nn.MultiheadAttention is opaque to thop and
    silently reports 0 FLOPs). Parameter count is identical to
    nn.MultiheadAttention: 3 in-projections + 1 out-projection, each E->E
    with bias, i.e. 4E^2 + 4E."""
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = self.head_dim ** -0.5

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * self.scale, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class GenericAttnLayer(nn.Module):
    """Same transformer-layer shell as CustomDeiTLayer, but takes any attn module."""
    def __init__(self, attn_module, embed_dim, mlp_ratio=4., drop_path=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = attn_module
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden), nn.GELU(), nn.Linear(mlp_hidden, embed_dim)
        )

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class GenericHybridModel(nn.Module):
    """Same CNN backbone / embed / pooling / classifier as HybridStudentModel and
    MHANetBaseline, so the comparison isolates the attention mechanism only."""
    def __init__(self, num_classes, attn_factory, embed_dim=768, num_layers=GLOBAL_NUM_LAYERS,
                 grid_size=3, drop_path_rate=0.1):
        super().__init__()
        self.resnet = ResNet18_Features()
        self.densenet = DenseNet121_Features()
        self.grid_size = grid_size
        concat_channels = 512 + 1024

        self.layers = nn.ModuleList([
            GenericAttnLayer(attn_factory(), embed_dim, drop_path=drop_path_rate)
            for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(concat_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        r = self.resnet(x)
        d = self.densenet(x)
        feats = torch.cat((r, d), dim=1)
        if self.grid_size != feats.shape[-1]:
            feats = F.adaptive_avg_pool2d(feats, (self.grid_size, self.grid_size))
        b, c, h, w = feats.shape
        feats = feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(feats)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)

In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, alpha=0.5, temperature=3.0):
        super(DistillationLoss, self).__init__()
        self.alpha = alpha
        self.temperature = temperature
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, ground_truth):
        hard_loss = self.ce_loss(student_logits, ground_truth)
        soft_loss = self.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=1),
            F.softmax(teacher_logits / self.temperature, dim=1)
        ) * (self.temperature ** 2)
        return self.alpha * soft_loss + (1 - self.alpha) * hard_loss


def evaluate_model(model, data_loader, criterion, return_predictions=False):
    model.eval()
    correct, total, test_loss = 0, 0, 0.0
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            if return_predictions:
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(predicted.cpu().numpy())
    accuracy = 100 * correct / total
    avg_loss = test_loss / len(data_loader)
    if return_predictions:
        return accuracy, avg_loss, np.array(all_labels), np.array(all_preds)
    return accuracy, avg_loss


def train_model_with_distillation(student_model, teacher_models, train_loader, val_loader,
                                   distillation_criterion, optimizer, num_epochs=1,
                                   restore_best_checkpoint=True):
    
    train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []
    best_val_loss = float('inf')
    best_val_acc_at_best_loss = None
    best_state = None
    for epoch in range(num_epochs):
        student_model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            student_outputs = student_model(images)
            with torch.no_grad():
                teacher_logits = [teacher(images) for teacher in teacher_models]
                combined_teacher_logits = sum(teacher_logits) / len(teacher_logits)
            loss = distillation_criterion(student_outputs, combined_teacher_logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)
        train_accuracy, _ = evaluate_model(student_model, train_loader, distillation_criterion.ce_loss)
        val_accuracy, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, '
              f'Train Acc: {train_accuracy:.2f}%, Val Acc: {val_accuracy:.2f}%')
        if restore_best_checkpoint and val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc_at_best_loss = val_accuracy
            best_state = {k: v.detach().cpu().clone() for k, v in student_model.state_dict().items()}
    if restore_best_checkpoint and best_state is not None:
        student_model.load_state_dict(best_state)
        print(f'  [best checkpoint] Restored weights from best Val Loss: {best_val_loss:.4f} '
              f'(Val Acc at that point: {best_val_acc_at_best_loss:.2f}%)')
    elif not restore_best_checkpoint:
        print(f'  [no restoration] Keeping final-epoch weights '
              f'(Val Acc: {val_accuracies[-1]:.2f}%, Val Loss: {val_losses[-1]:.4f})')
    return student_model, train_losses, val_losses, train_accuracies, val_accuracies


def train_model_plain(model, train_loader, val_loader, criterion, optimizer, num_epochs=20):
    
    train_losses, val_losses, train_accuracies, val_accuracies = [], [], [], []
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)
        train_accuracy, _ = evaluate_model(model, train_loader, criterion)
        val_accuracy, val_loss = evaluate_model(model, val_loader, criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%, "
              f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")
    return train_losses, val_losses, train_accuracies, val_accuracies


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_custom_parameters(model, exclude_list=None):
    
    exclude_list = exclude_list or []
    excluded_params = set(p for submodule in exclude_list for p in submodule.parameters())
    return sum(p.numel() for p in model.parameters() if p.requires_grad and p not in excluded_params)


def measure_inference_time(model, dummy_input, n_runs=50, device='cuda'):
    model.eval()
    with torch.no_grad():
        for _ in range(5):  
            _ = model(dummy_input)
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy_input)
    if device == 'cuda':
        torch.cuda.synchronize()
    end = time.time()
    return (end - start) / n_runs * 1000  


def measure_memory(model, dummy_input, device='cuda'):
    if device != 'cuda':
        return None
    torch.cuda.reset_peak_memory_stats(device)
    model.eval()
    with torch.no_grad():
        _ = model(dummy_input)
    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)  


def measure_inference_time_and_memory(model, data_loader, device='cuda'):
  
    model.eval()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize()

    start_time = time.time()
    n_images = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            n_images += images.size(0)
            outputs = model(images)
    if device == 'cuda':
        torch.cuda.synchronize()
    end_time = time.time()

    total_time = end_time - start_time
    avg_time_per_image_ms = (total_time / n_images) * 1000
    peak_memory = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device == 'cuda' else None

    return peak_memory, total_time, avg_time_per_image_ms


def measure_efficiency_isolated(models_dict, dummy_input, device='cuda', n_runs=50, warmup=5):
   
    results = {}
    original_devices = {name: next(m.parameters()).device for name, m in models_dict.items()}

    for name, model in models_dict.items():
        for other_name, other_model in models_dict.items():
            if other_name != name:
                other_model.to('cpu')
        torch.cuda.empty_cache()
        gc.collect()

        model.to(device).eval()

        with torch.no_grad():
            for _ in range(warmup):
                _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()

        if device == 'cuda':
            torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()
        peak_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device == 'cuda' else None

        if device == 'cuda':
            torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            for _ in range(n_runs):
                _ = model(dummy_input)
        if device == 'cuda':
            torch.cuda.synchronize()
        inf_time = (time.time() - start) / n_runs * 1000  

        results[name] = {
            'Inference Time (ms)': round(inf_time, 4),
            'Peak Memory (MB)': round(peak_mem, 4) if peak_mem is not None else None,
        }
        print(f"[{name}] measured in isolation -> Time: {inf_time:.4f} ms, "
              f"Memory: {peak_mem:.2f} MB" if peak_mem is not None else
              f"[{name}] measured in isolation -> Time: {inf_time:.4f} ms")

    for name, model in models_dict.items():
        model.to(original_devices[name])

    return pd.DataFrame(results).T


In [ ]:
_THOP_ARTIFACT_SUFFIXES = ('.total_ops', '.total_params')
_THOP_ARTIFACT_NAMES = ('total_ops', 'total_params')

def _strip_thop_artifacts(state_dict):
    
    return {
        k: v for k, v in state_dict.items()
        if k not in _THOP_ARTIFACT_NAMES and not k.endswith(_THOP_ARTIFACT_SUFFIXES)
    }


def checkpoint_path(current_dataset, model_name):
    safe_name = (model_name.replace(" ", "_").replace("(", "").replace(")", "")
                 .replace("/", "-").replace("\\", "-").replace(":", "-"))
    return f"{OUTPUT_ROOT}/{current_dataset}_{safe_name}_checkpoint.pth"


# Save/load checkpoints ONLY for the main LoRaS-CT vs MHA-Net comparison.
MAIN_COMPARISON_CHECKPOINT_MODELS = {"LoRaS-CT", "MHA-Net (baseline)"}

def save_model_checkpoint(current_dataset, model_name, model, result_dict, history):
    """Save checkpoints only for the main LoRaS-CT vs MHA-Net comparison."""
    if model_name not in MAIN_COMPARISON_CHECKPOINT_MODELS:
        return

    path = checkpoint_path(current_dataset, model_name)
    torch.save({
        'model_state_dict': _strip_thop_artifacts(model.state_dict()),
        'result': result_dict,
        'history': history,
    }, path)
    print(f"  [checkpoint saved] {path}")


def load_model_checkpoint(current_dataset, model_name, model):
    """Load checkpoints only for the main LoRaS-CT vs MHA-Net comparison."""
    if model_name not in MAIN_COMPARISON_CHECKPOINT_MODELS:
        return None

    path = checkpoint_path(current_dataset, model_name)
    if FORCE_RETRAIN or not os.path.exists(path):
        return None
    ckpt = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
    model.load_state_dict(_strip_thop_artifacts(ckpt['model_state_dict']))
    print(f"  [checkpoint found] Skipping training for '{model_name}' - loaded from {path}")
    return ckpt['result'], ckpt['history']


def save_comparison_metrics_to_checkpoint(current_dataset, model_name, comparison_metrics):
    """Add Table 3 metrics to an existing main-comparison checkpoint only."""
    if model_name not in MAIN_COMPARISON_CHECKPOINT_MODELS:
        return
    path = checkpoint_path(current_dataset, model_name)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Comparison checkpoint not found: {path}")

    ckpt = torch.load(path, map_location='cpu')
    ckpt['comparison_metrics'] = {
        key: float(value) for key, value in comparison_metrics.items()
    }
    torch.save(ckpt, path)
    print(f"  [Table 3 metrics saved in checkpoint] {path}")


In [ ]:
class LowRankSparseMultiheadAttention_Visualizable(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, sparsity_ratio=0.5):
        super(LowRankSparseMultiheadAttention_Visualizable, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.rank = rank
        self.sparsity_ratio = sparsity_ratio

        self.q_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.q_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])
        self.k_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.k_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])
        self.v_lows = nn.ModuleList([nn.Linear(embed_dim, rank, bias=False) for _ in range(num_heads)])
        self.v_highs = nn.ModuleList([nn.Linear(rank, embed_dim, bias=False) for _ in range(num_heads)])

        self.out_proj = nn.Linear(embed_dim * num_heads, embed_dim)
        self.scale = embed_dim ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        
        batch_size, num_heads, seq_length, _ = attn_scores.size()
        if seq_length == 1:
            return attn_scores, torch.ones_like(attn_scores)
        num_to_keep = int(sparsity_ratio * seq_length)
        top_scores, _ = torch.topk(attn_scores, k=num_to_keep, dim=-1)
        threshold = top_scores.min(dim=-1, keepdim=True)[0]
        sparse_mask = attn_scores >= threshold
        sparse_attn_scores = attn_scores.masked_fill(~sparse_mask, float('-inf'))
        return sparse_attn_scores, sparse_mask

    def forward(self, x):
        batch_size, seq_length, embed_dim = x.size()

        q, k, v = [], [], []
        for i in range(self.num_heads):
            q.append(self.q_highs[i](self.q_lows[i](x)).view(batch_size, seq_length, embed_dim))
            k.append(self.k_highs[i](self.k_lows[i](x)).view(batch_size, seq_length, embed_dim))
            v.append(self.v_highs[i](self.v_lows[i](x)).view(batch_size, seq_length, embed_dim))

        q = torch.stack(q, dim=1)
        k = torch.stack(k, dim=1)
        v = torch.stack(v, dim=1)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        sparse_attn_scores, sparse_mask = self.sparse_attention(attn_scores, self.sparsity_ratio)
        attn_probs = F.softmax(sparse_attn_scores, dim=-1)
        attn_output = torch.matmul(attn_probs, v)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_length, self.num_heads * embed_dim)

        output = self.out_proj(attn_output)
        return output, sparse_attn_scores, sparse_mask


class CustomDeiTLayer_Visualizable(nn.Module):
    def __init__(self, embed_dim, num_heads, rank, mlp_ratio=4., drop_path=0.1, sparsity_ratio=0.5):
        super(CustomDeiTLayer_Visualizable, self).__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = LowRankSparseMultiheadAttention_Visualizable(embed_dim, num_heads, rank, sparsity_ratio)
        self.drop_path = nn.Identity()
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, embed_dim),
        )

    def forward(self, x):
        attn_output, attn_scores, sparse_mask = self.attn(self.norm1(x))
        x = x + self.drop_path(attn_output)
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x, attn_scores, sparse_mask


class HybridStudentModel_Visualizable(nn.Module):
    
    def __init__(self, num_classes, embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS, rank=GLOBAL_RANK,
                 drop_path_rate=0.1, sparsity_ratio=0.5):
        super(HybridStudentModel_Visualizable, self).__init__()
        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer_Visualizable(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                                          sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(2000, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        x = self.deit_embed(x)
        for layer in self.deit_layers:
            x, attn_scores, sparse_mask = layer(x)
        x = self.norm(x)
        x = x.squeeze(1)
        x = self.classifier(x)
        return x, attn_scores, sparse_mask


def visualize_attention_and_sparsity(attn_scores, sparse_mask, head=0):
    attn_scores = attn_scores[0, head].cpu().detach().numpy()
    sparse_mask = sparse_mask[0, head].cpu().detach().numpy()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5), dpi=150)
    ax1.imshow(attn_scores, cmap='viridis')
    ax1.set_title(f'Attention Scores (Head {head})')
    ax1.set_xlabel('Key Position')
    ax1.set_ylabel('Query Position')

    ax2.imshow(sparse_mask, cmap='gray')
    ax2.set_title(f'Sparsity Mask (Head {head})')
    ax2.set_xlabel('Key Position')
    ax2.set_ylabel('Query Position')
    plt.show()


def forward_with_visualization(model, data_loader, head=1):
    model.eval()
    with torch.no_grad():
        for batch in data_loader:
            images = batch.cuda() if isinstance(batch, torch.Tensor) else batch[0].cuda()
            outputs, attn_scores, sparse_mask = model(images)
            visualize_attention_and_sparsity(attn_scores, sparse_mask, head=head)
            break


demo_batch_size = 32
demo_embed_dim = 768
demo_num_classes = 8  

demo_hybrid_model = HybridStudentModel_Visualizable(num_classes=demo_num_classes, embed_dim=demo_embed_dim).cuda()

demo_loader_1 = torch.utils.data.DataLoader(torch.randn(demo_batch_size, 1, 2000), batch_size=demo_batch_size)
forward_with_visualization(demo_hybrid_model, demo_loader_1, head=1)

demo_seq_len = 16  
demo_loader_2 = torch.utils.data.DataLoader(torch.randn(demo_batch_size, demo_seq_len, 2000), batch_size=demo_batch_size)
forward_with_visualization(demo_hybrid_model, demo_loader_2, head=1)

In [ ]:
from scipy import stats

N_SEEDS = 1 if QUICK_TEST_MODE else 3
SEED_LIST = [42, 123, 2024][:N_SEEDS]
STAT_NUM_EPOCHS = GLOBAL_NUM_EPOCHS  

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_split(dataset_path, seed, use_class_balancing=True):
    
    full_ds = datasets.ImageFolder(dataset_path)
    group_split = patient_group_split(full_ds, seed=seed) if use_class_balancing else None

    if group_split is not None:
        train_idx, val_idx, test_idx = group_split
        train_ds = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
        val_ds = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
        test_ds = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
    else:
        # Plain random split (no stratification) for datasets without a parseable
        # patient/group ID - Kather5k, CRC7k - matching the reference
        # implementation this was checked against.
        from sklearn.model_selection import train_test_split as _tts
        _all_labels = np.asarray(full_ds.targets)
        _all_idx = np.arange(len(full_ds))
        trainval_idx, test_idx = _tts(_all_idx, test_size=0.2, random_state=seed)
        train_idx, val_idx = _tts(
            trainval_idx, test_size=0.1, random_state=seed
        )
        train_ds = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
        val_ds = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
        test_ds = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)

    train_ds = quick_subset(train_ds, QUICK_TEST_MAX_TRAIN)
    val_ds = quick_subset(val_ds, QUICK_TEST_MAX_VAL)
    test_ds = quick_subset(test_ds, QUICK_TEST_MAX_TEST)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    vl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    tel = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return tl, vl, tel


In [ ]:

import sympy as sp

E, N, H, R = sp.symbols('E N H R', positive=True, integer=True)

mha_params  = 4*E**2 + 4*E
mha_flops   = 4*N*E**2 + 2*N**2*E          

lora_params = 4*E*R + R**2
lora_flops  = 4*N*E*R + 2*N**2*R + N*R**2  

print("Standard MHA   params:", mha_params, "   FLOPs:", mha_flops)
print("LoRa-SMHA      params:", lora_params, "   FLOPs:", lora_flops)

reduction_params = sp.simplify(mha_params / lora_params)
print("\nSymbolic params ratio (MHA / LoRa-SMHA):", reduction_params)

E_val, H_val, R_val = 768, GLOBAL_NUM_HEADS, GLOBAL_RANK
rows = []
for N_val in [9, 16, 25, 49, 100]:  
    mp = int(mha_params.subs({E: E_val}))
    mf = int(mha_flops.subs({E: E_val, N: N_val}))
    lp = int(lora_params.subs({E: E_val, R: R_val}))
    lf = int(lora_flops.subs({E: E_val, N: N_val, R: R_val}))
    rows.append({
        'N (tokens)': N_val, 'MHA Params': mp, 'MHA FLOPs': mf,
        'LoRa-SMHA Params': lp, 'LoRa-SMHA FLOPs': lf,
        'Param Reduction': f"{mp/lp:.2f}x", 'FLOPs Reduction': f"{mf/lf:.2f}x",
    })
complexity_theory_df = pd.DataFrame(rows)
print(f"\nTable: Theoretical complexity, single attention layer, E={E_val}, H={H_val}, R={R_val}\n")
print(complexity_theory_df.to_string(index=False))


In [ ]:

attn_lora = LowRankSparseMultiheadAttention(embed_dim=768, num_heads=GLOBAL_NUM_HEADS, rank=GLOBAL_RANK, sparsity_ratio=0.5).cuda()
attn_mha  = StandardMHAWrapper(embed_dim=768, num_heads=GLOBAL_NUM_HEADS).cuda()

dummy_tokens = torch.randn(1, 9, 768).cuda()  

flops_lora, params_lora = profile(attn_lora, inputs=(dummy_tokens,), verbose=False)
flops_mha,  params_mha  = profile(attn_mha,  inputs=(dummy_tokens,), verbose=False)

time_lora = measure_inference_time(attn_lora, dummy_tokens, n_runs=100)
time_mha  = measure_inference_time(attn_mha, dummy_tokens, n_runs=100)
mem_lora  = measure_memory(attn_lora, dummy_tokens)
mem_mha   = measure_memory(attn_mha, dummy_tokens)

empirical_complexity_df = pd.DataFrame({
    'Standard MHA': {'Params': int(params_mha), 'FLOPs': int(flops_mha),
                      'Time (ms)': round(time_mha, 4), 'Peak Memory (MB)': round(mem_mha, 4)},
    'LoRa-SMHA':    {'Params': int(params_lora), 'FLOPs': int(flops_lora),
                      'Time (ms)': round(time_lora, 4), 'Peak Memory (MB)': round(mem_lora, 4)},
}).T
print(f"Table: Empirical (thop-profiled) complexity, isolated attention layer, N=9, E=768, H={GLOBAL_NUM_HEADS}, R={GLOBAL_RANK}\n")
print(empirical_complexity_df.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

x = np.arange(2)
width = 0.35
axes[0].bar(x - width/2, [params_mha, params_lora], width, label='Params')
axes[0].bar(x + width/2, [flops_mha, flops_lora], width, label='FLOPs')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['Standard MHA', 'LoRa-SMHA'])
axes[0].set_yscale('log')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Attention-layer Params & FLOPs (N=9, E=768)')
axes[0].legend()

N_range = np.array([9, 16, 25, 49, 100, 196])
mha_flops_vals = [int(mha_flops.subs({E: 768, N: int(n)})) for n in N_range]
lora_flops_vals = [int(lora_flops.subs({E: 768, N: int(n), R: GLOBAL_RANK})) for n in N_range]
axes[1].plot(N_range, mha_flops_vals, marker='o', label='Standard MHA')
axes[1].plot(N_range, lora_flops_vals, marker='s', label=f'LoRa-SMHA (R={GLOBAL_RANK})')
axes[1].axvline(9, color='gray', linestyle='--', alpha=0.5)
axes[1].text(9, max(mha_flops_vals)*0.9, 'this paper\n(grid=3)', fontsize=8, ha='left')
axes[1].set_xlabel('N (tokens)')
axes[1].set_ylabel('FLOPs')
axes[1].set_title('Theoretical FLOPs vs Sequence Length')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/complexity_analysis.png', dpi=150)
plt.show()


In [ ]:
class GradCAM:
    """Standard Grad-CAM applied to a CNN feature-extractor submodule. Works on any
    nn.Module target layer that outputs a (B, C, H, W) spatial map."""
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self.activations = out.detach()

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(output.argmax(dim=1).item())
        self.model.zero_grad()
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    def remove(self):
        self._fwd.remove()
        self._bwd.remove()


In [ ]:
class FullRankSparseAttention(nn.Module):
    
    def __init__(self, embed_dim, num_heads, sparsity_ratio=0.5):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.sparsity_ratio = sparsity_ratio
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.scale = embed_dim ** -0.5

    def sparse_attention(self, attn_scores, sparsity_ratio):
        
        b, h, t, _ = attn_scores.size()
        if t == 1:
            return attn_scores
        k = max(1, int(sparsity_ratio * t))
        top, _ = torch.topk(attn_scores, k=k, dim=-1)
        thresh = top.min(dim=-1, keepdim=True)[0]
        mask = attn_scores >= thresh
        return attn_scores.masked_fill(~mask, float('-inf'))

    def forward(self, x):
        B, N, E = x.shape
        Q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        scores = self.sparse_attention(scores, self.sparsity_ratio)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, E)
        return self.out_proj(out)


class SingleBackboneHybridModel(nn.Module):
    """Ablation: keeps the full LoRa-SMHA transformer, removes one of the two CNN branches."""
    def __init__(self, num_classes, backbone='resnet', embed_dim=768, num_heads=GLOBAL_NUM_HEADS, num_layers=GLOBAL_NUM_LAYERS,
                 rank=GLOBAL_RANK, sparsity_ratio=0.5, grid_size=3, drop_path_rate=0.1):
        super().__init__()
        assert backbone in ('resnet', 'densenet')
        if backbone == 'resnet':
            self.backbone = ResNet18_Features()
            in_channels = 512
        else:
            self.backbone = DenseNet121_Features()
            in_channels = 1024
        self.grid_size = grid_size
        self.deit_layers = nn.ModuleList([
            CustomDeiTLayer(embed_dim, num_heads, rank, drop_path=drop_path_rate,
                             sparsity_ratio=sparsity_ratio) for _ in range(num_layers)
        ])
        self.deit_embed = nn.Linear(in_channels, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        if self.grid_size != feats.shape[-1]:
            feats = F.adaptive_avg_pool2d(feats, (self.grid_size, self.grid_size))
        b, c, h, w = feats.shape
        feats = feats.view(b, c, h * w).permute(0, 2, 1)
        x = self.deit_embed(feats)
        for layer in self.deit_layers:
            x = layer(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.classifier(x)


### 3.2 Pathology foundation models (UNI / CONCH / Virchow) — R3 fix merged in

This replaces the original Section 18 encoder-loading cell with the **fixed** version from the R3 standalone notebook: CONCH is loaded via its own `conch` package (not `transformers.AutoModel`), and Virchow's raw token-sequence output is pooled into the documented 2560-dim embedding. One-time install needed (internet required):

In [ ]:
!pip install -q git+https://github.com/Mahmoodlab/CONCH.git

In [ ]:
FOUNDATION_MODEL_EPOCHS = GLOBAL_NUM_EPOCHS  

class _CONCHImageEncoderWrapper(nn.Module):
  
    def __init__(self, conch_model):
        super().__init__()
        self.conch_model = conch_model

    def forward(self, x):
        return self.conch_model.encode_image(x, proj_contrast=False, normalize=False)


class _VirchowEncoderWrapper(nn.Module):
    
    def __init__(self, vit_model):
        super().__init__()
        self.vit_model = vit_model

    def forward(self, x):
        output = self.vit_model(x)            
        class_token = output[:, 0]             
        patch_tokens = output[:, 1:]            
        return torch.cat([class_token, patch_tokens.mean(1)], dim=-1)  


def load_foundation_encoder(name):
    
    if name == 'UNI':
        import timm
        encoder = timm.create_model(
            "hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, num_classes=0
        )
        embed_dim = 1024
    elif name == 'CONCH':
        from conch.open_clip_custom import create_model_from_pretrained
        conch_model, _ = create_model_from_pretrained(
            'conch_ViT-B-16', "hf_hub:MahmoodLab/conch", hf_auth_token=globals().get('hf_token')
        )
        encoder = _CONCHImageEncoderWrapper(conch_model)
        embed_dim = 512
    elif name == 'Virchow':
        import timm
        vit_model = timm.create_model(
            "hf-hub:paige-ai/Virchow", pretrained=True,
            mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU
        )
        encoder = _VirchowEncoderWrapper(vit_model)
        embed_dim = 2560  
    elif name == 'HIPT':
        raise NotImplementedError("Point this at your local HIPT ViT-S/16 checkpoint (see comment above).")
    else:
        raise ValueError(f"Unknown foundation model: {name}")

    encoder = encoder.cuda().eval()
    for p in encoder.parameters():
        p.requires_grad = False
    return encoder, embed_dim


class LinearProbeHead(nn.Module):
    """Frozen encoder + trainable linear classification head - the standard evaluation
    protocol for foundation-model patch encoders."""
    def __init__(self, encoder, embed_dim, num_classes):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            feats = self.encoder(x)
            if isinstance(feats, (tuple, list)):
                feats = feats[0]
        return self.head(feats)


class CLAMAttentionHead(nn.Module):
    
    def __init__(self, encoder, embed_dim, num_classes, attn_dim=256):
        super().__init__()
        self.encoder = encoder
        self.attn_V = nn.Linear(embed_dim, attn_dim)
        self.attn_U = nn.Linear(embed_dim, attn_dim)
        self.attn_w = nn.Linear(attn_dim, 1)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            feats = self.encoder(x)
            if isinstance(feats, (tuple, list)):
                feats = feats[0]
        a = torch.tanh(self.attn_V(feats)) * torch.sigmoid(self.attn_U(feats))
        gate = torch.sigmoid(self.attn_w(a))
        gated_feats = feats * gate
        return self.classifier(gated_feats)

Hugging Face login (gated models) — run once, before the dataset loop:

In [ ]:

from huggingface_hub import login
import getpass

hf_login_ok = False
hf_token = None

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except ModuleNotFoundError:
    pass  
except Exception as e:
    print(f"Kaggle Secrets lookup failed ({type(e).__name__}: {e}) -- falling back.")

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    try:
        from huggingface_hub import HfApi
        HfApi().whoami()  
        hf_login_ok = True
        print("Reusing existing cached Hugging Face login (from huggingface-cli login).")
    except Exception:
        pass  

if not hf_login_ok and not hf_token:
    print("No cached login, HF_TOKEN env var, or Kaggle secret found.")
    print("Paste your Hugging Face token below (from https://huggingface.co/settings/tokens)")
    print("and press Enter - your typing will be hidden.")
    hf_token = getpass.getpass("HF token: ")

if not hf_login_ok:
    try:
        login(token=hf_token)
        hf_login_ok = True
        print("Hugging Face login successful.")
    except Exception as e:
        print(f"Hugging Face login FAILED: {type(e).__name__}: {e}")
        print("Foundation model downloads (UNI/CONCH/Virchow) below will be skipped until this is fixed --")
        print("see the setup steps in this cell's comments.")


In [ ]:
from huggingface_hub import HfApi

api = HfApi()

if not hf_login_ok:
    print("Not logged in -fix the cell above first (check your HF_TOKEN Kaggle Secret).")
else:
    try:
        who = api.whoami()
        print(f"Logged in as: {who['name']}  (email verified: {who.get('email', 'unknown')})\n")
    except Exception as e:
        print(f"Logged in, but whoami() failed: {type(e).__name__}: {e}\n")

    gated_repos = ['MahmoodLab/uni', 'MahmoodLab/CONCH', 'paige-ai/Virchow']
    print("Gated model access status:\n")
    for repo_id in gated_repos:
        try:
            api.model_info(repo_id)
            print(f"  [ACCESS GRANTED]  {repo_id}")
        except Exception as e:
            reason = type(e).__name__
            if 'Gated' in reason or '403' in str(e):
                print(f"  [PENDING/DENIED]  {repo_id}  -- request not yet approved (or was denied)")
            else:
                print(f"  [ERROR]           {repo_id}  -- {reason}: {e}")
    print("\nOnce all three show [ACCESS GRANTED], Section 18 below will download and run for real.")


In [ ]:
def run_pipeline(CURRENT_DATASET, DATASET_CFG):
    
    _weight_decay = DATASET_CFG.get('weight_decay', DEFAULT_WEIGHT_DECAY)
    _restore_best_checkpoint = DATASET_CFG.get('restore_best_checkpoint', DEFAULT_RESTORE_BEST_CHECKPOINT)
    _use_class_balancing = DATASET_CFG.get('use_class_balancing', DEFAULT_USE_CLASS_BALANCING)
    print(f"[{CURRENT_DATASET}] Weight decay: {_weight_decay}")
    print(f"[{CURRENT_DATASET}] Restore best checkpoint: {_restore_best_checkpoint}")
    print(f"[{CURRENT_DATASET}] Class-balanced group splitting: {_use_class_balancing}")
    is_presplit = "test_path" in DATASET_CFG
    _path_desc = (f"train_val={DATASET_CFG['train_val_path']} | test={DATASET_CFG['test_path']}"
                  if is_presplit else DATASET_CFG["path"])
    print(f"\n{'#'*80}\n# RUNNING PIPELINE FOR DATASET: {CURRENT_DATASET}\n# Path: {_path_desc}\n{'#'*80}\n")
    img_dir = f"images/{CURRENT_DATASET}"
    os.makedirs(img_dir, exist_ok=True)



    if is_presplit and CURRENT_DATASET == "LC25000":
        train_dataset, val_dataset, test_dataset, num_classes, dataset = \
            build_lc25000_grouped_split(DATASET_CFG, train_val_transform, test_transform, seed=42)
        dataset_path = DATASET_CFG["train_val_path"]  
    elif is_presplit:
        dataset_path = DATASET_CFG["train_val_path"]
        train_val_dataset = datasets.ImageFolder(DATASET_CFG["train_val_path"])
        test_dataset = datasets.ImageFolder(DATASET_CFG["test_path"])

        assert train_val_dataset.classes == test_dataset.classes, (
            f"[{CURRENT_DATASET}] Class mismatch between train/val and test folders: "
            f"{train_val_dataset.classes} vs {test_dataset.classes}. "
            f"Both folders must contain identically-named class sub-folders."
        )
        dataset = train_val_dataset  
        num_classes = len(dataset.classes)

        
        from sklearn.model_selection import train_test_split as _tts
        _all_labels = np.asarray(train_val_dataset.targets)
        _all_idx = np.arange(len(train_val_dataset))
        train_idx, val_idx = _tts(
            _all_idx,
            test_size=0.1,
            random_state=42
        )

        # Use separate dataset instances so training and validation can keep
        # their own transforms without overwriting each other.
        train_dataset = Subset(
            datasets.ImageFolder(DATASET_CFG["train_val_path"], transform=train_val_transform),
            train_idx
        )
        val_dataset = Subset(
            datasets.ImageFolder(DATASET_CFG["train_val_path"], transform=test_transform),
            val_idx
        )
        test_dataset = datasets.ImageFolder(DATASET_CFG["test_path"], transform=test_transform)
    else:
        dataset_path = DATASET_CFG["path"]
        dataset = datasets.ImageFolder(dataset_path)
        num_classes = len(dataset.classes)

        group_split = patient_group_split(dataset, seed=42) if _use_class_balancing else None
        if group_split is not None:
            train_idx, val_idx, test_idx = group_split
            print(f"[{CURRENT_DATASET}] Patient/slide IDs detected -- using the 500-restart "
                  f"balanced group-fold split (patient-disjoint AND class-balanced).")
            train_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
            val_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
            test_dataset = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
        else:
            # Plain random split (no stratification) for datasets without a parseable
            # patient/group ID -  Kather5k, CRC7k - matching the reference
            # implementation this was checked against.
            from sklearn.model_selection import train_test_split as _tts
            _all_labels = np.asarray(dataset.targets)
            _all_idx = np.arange(len(dataset))
            trainval_idx, test_idx = _tts(_all_idx, test_size=0.2, random_state=42)
            train_idx, val_idx = _tts(
                trainval_idx, test_size=0.1, random_state=42
            )
            train_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
            val_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
            test_dataset = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)

    train_dataset = quick_subset(train_dataset, QUICK_TEST_MAX_TRAIN)
    val_dataset = quick_subset(val_dataset, QUICK_TEST_MAX_VAL)
    test_dataset = quick_subset(test_dataset, QUICK_TEST_MAX_TEST)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    print(f"[{CURRENT_DATASET}] Classes: {dataset.classes}")
    print(f"[{CURRENT_DATASET}] Train/Val/Test sizes: {len(train_dataset)}/{len(val_dataset)}/{len(test_dataset)}"
          + ("  [QUICK_TEST_MODE: capped]" if QUICK_TEST_MODE else ""))



    teacher_vit = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_deit = create_model('deit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_swin = create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes).cuda()

    
    for _t in (teacher_vit, teacher_deit, teacher_swin):
        _t.eval()
        for _p in _t.parameters():
            _p.requires_grad = False

    teacher_models = [teacher_vit, teacher_deit, teacher_swin]

    def run_full_comparison(num_classes, teacher_models, train_loader, val_loader, test_loader,
                             num_epochs=20, device='cuda', alpha=0.5, temperature=3.0,
                             lr=0.01, momentum=0.9, grid_size=3):

        models_to_compare = {
            'LoRaS-CT': HybridStudentModel(num_classes, grid_size=grid_size).to(device),
            'MHA-Net (baseline)': MHANetBaseline(num_classes, num_heads=8, num_layers=2, grid_size=grid_size).to(device),
        }

        results = {}
        trained_models = {}

        for name, model in models_to_compare.items():
            print(f"\n{'='*60}")
            print(f"Training: {name}  (grid_size={grid_size} -> N={grid_size*grid_size} tokens)")
            print(f"{'='*60}")

            cached = load_model_checkpoint(CURRENT_DATASET, name, model)
            if cached is not None:
                cached_result, cached_history = cached
                trained_models[name] = {'model': model, **cached_history}
                
                dummy_input = torch.randn(1, 3, 224, 224).to(device)
                params = count_parameters(model)
                flops, _ = profile(model, inputs=(dummy_input,), verbose=False)
                inf_time = measure_inference_time(model, dummy_input, device=device)
                memory = measure_memory(model, dummy_input, device=device)
                results[name] = {
                    'Test Accuracy (%)': cached_result['Test Accuracy (%)'],
                    'Params (M)': params / 1e6,
                    'FLOPs (G)': flops / 1e9,
                    'Inference Time (ms)': inf_time,
                    'Peak Memory (MB)': memory,
                }
                continue

            criterion = DistillationLoss(alpha=alpha, temperature=temperature)
            optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=_weight_decay)

            trained_model, train_losses, val_losses, train_accs, val_accs = \
                train_model_with_distillation(
                    model, teacher_models, train_loader, val_loader,
                    criterion, optimizer, num_epochs=num_epochs,
                    restore_best_checkpoint=_restore_best_checkpoint
                )

            test_accuracy, test_loss = evaluate_model(trained_model, test_loader, criterion.ce_loss)
            print(f"[{name}] Test Accuracy: {test_accuracy:.2f}%, Test Loss: {test_loss:.4f}")

            dummy_input = torch.randn(1, 3, 224, 224).to(device)
            params = count_parameters(trained_model)
            flops, _ = profile(trained_model, inputs=(dummy_input,), verbose=False)
            inf_time = measure_inference_time(trained_model, dummy_input, device=device)
            memory = measure_memory(trained_model, dummy_input, device=device)

            results[name] = {
                'Test Accuracy (%)': test_accuracy,
                'Params (M)': params / 1e6,
                'FLOPs (G)': flops / 1e9,
                'Inference Time (ms)': inf_time,
                'Peak Memory (MB)': memory,
            }
            history = {'train_losses': train_losses, 'val_losses': val_losses,
                       'train_accs': train_accs, 'val_accs': val_accs}
            trained_models[name] = {'model': trained_model, **history}

            
            checkpoint_result = {'Test Accuracy (%)': test_accuracy}
            save_model_checkpoint(CURRENT_DATASET, name, trained_model, checkpoint_result, history)

        print(f"\n{'='*80}")
        print(f"FINAL COMPARISON -- {CURRENT_DATASET}")
        print(f"{'='*80}")
        metrics = ['Test Accuracy (%)', 'Params (M)', 'FLOPs (G)', 'Inference Time (ms)', 'Peak Memory (MB)']
        header = f"{'Metric':<25}" + "".join(f"{name:<25}" for name in results.keys())
        print(header)
        for metric in metrics:
            row = f"{metric:<25}"
            for name in results.keys():
                row += f"{results[name][metric]:<25.4f}"
            print(row)

        return results, trained_models


    comparison_results, comparison_trained_models = run_full_comparison(
        num_classes=num_classes,
        teacher_models=teacher_models,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        num_epochs=GLOBAL_NUM_EPOCHS,
        device='cuda',
        grid_size=3,
    )

    loras_ct_model = comparison_trained_models['LoRaS-CT']['model'].cuda()
    mha_net_model = comparison_trained_models['MHA-Net (baseline)']['model'].cuda()

    print(f'LoRaS-CT (Algorithm-1-accurate) : {count_parameters(loras_ct_model):,} parameters')
    print(f'MHA-Net (baseline)         : {count_parameters(mha_net_model):,} parameters')
    print(f'Teacher ViT                : {count_parameters(teacher_vit):,} parameters')
    print(f'Teacher DeiT                : {count_parameters(teacher_deit):,} parameters')
    print(f'Teacher Swin Transformer   : {count_parameters(teacher_swin):,} parameters')


    _eff_embed_dim, _eff_num_heads, _eff_num_layers, _eff_grid_size = 768, GLOBAL_NUM_HEADS, GLOBAL_NUM_LAYERS, 3
    _eff_N = _eff_grid_size * _eff_grid_size
    _eff_dummy = torch.randn(1, 3, 224, 224).to('cuda')

    mha_net_model.to('cpu')
    loras_ct_model.to('cpu')
    torch.cuda.empty_cache()
    gc.collect()

    _efficiency_models = {
        'MHA-Net (baseline)': mha_net_model,
        'LoRaS-CT': loras_ct_model,
        'Linformer': GenericHybridModel(
            num_classes, attn_factory=lambda: LinformerAttention(_eff_embed_dim, _eff_num_heads, seq_len=_eff_N),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
        'Performer': GenericHybridModel(
            num_classes, attn_factory=lambda: PerformerAttention(_eff_embed_dim, _eff_num_heads),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
        'BigBird': GenericHybridModel(
            num_classes, attn_factory=lambda: BigBirdAttention(_eff_embed_dim, _eff_num_heads, block_size=64),
            embed_dim=_eff_embed_dim, num_layers=_eff_num_layers, grid_size=_eff_grid_size),
    }

    print(f"\n{'='*70}\nSection 8b: isolated efficiency measurement -- {CURRENT_DATASET}\n{'='*70}")
    isolated_efficiency_table = measure_efficiency_isolated(
        _efficiency_models, _eff_dummy, device='cuda', n_runs=100
    )
    print(isolated_efficiency_table.to_string())

    for _mname in ['MHA-Net (baseline)', 'LoRaS-CT']:
        comparison_results[_mname]['Inference Time (ms)'] = isolated_efficiency_table.loc[_mname, 'Inference Time (ms)']
        comparison_results[_mname]['Peak Memory (MB)'] = isolated_efficiency_table.loc[_mname, 'Peak Memory (MB)']

    del _efficiency_models
    torch.cuda.empty_cache()
    gc.collect()

    mha_net_model.to('cuda')
    loras_ct_model.to('cuda')



    def get_predictions_and_labels(model, data_loader, input_size=None):
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images, labels = images.cuda(), labels.cuda()
                if input_size is not None:
                    images = F.interpolate(images, size=input_size, mode='bilinear', align_corners=False)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.array(all_preds), np.array(all_labels)


    def get_predictions_and_labels_probs(model, data_loader):
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images, labels = images.cuda(), labels.cuda()
                outputs = model(images)
                all_preds.append(outputs.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        return np.vstack(all_preds), np.concatenate(all_labels)


    def plot_confusion_matrix(y_true, y_pred, class_names, save_path="images/confusion_matrix.png"):
        cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

        plt.figure(figsize=(6, 6), dpi=300)
        disp.plot(cmap=plt.cm.Blues, ax=plt.gca(), values_format='d', colorbar=False)
        for i in range(len(class_names)):
            for j in range(len(class_names)):
                plt.text(j, i, f'{cm[i, j]}', ha='center', va='center', fontsize=10, color='black',
                         fontweight='bold', bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.4'))
        plt.xticks(rotation=45, ha='right', fontsize=12)
        plt.yticks(fontsize=12)
        plt.xlabel('Predicted Labels', fontsize=12)
        plt.ylabel('True Labels', fontsize=12)
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()

        all_labels = list(range(len(class_names)))
        print("Accuracy:", accuracy_score(y_true, y_pred))
        print("Precision:", precision_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("Recall:", recall_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("F1 Score:", f1_score(y_true, y_pred, average='weighted', labels=all_labels, zero_division=0))
        print("\nClassification Report:\n", classification_report(
            y_true, y_pred, labels=all_labels, target_names=class_names, zero_division=0))


    def plot_roc_curve(y_true, y_scores, class_names, save_path="images/roc_curve.png"):
        n_classes = len(class_names)
        y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
        if n_classes == 2 and y_true_bin.shape[1] == 1:
            y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])
        plt.figure(figsize=(6, 5), dpi=300)
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=3, label=f'{class_names[i]} (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.title('Receiver Operating Characteristic')
        plt.legend(loc='lower right', fontsize=8)
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()


    def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies,
                              save_path="images/training_curves.png"):
        epochs = range(1, len(train_losses) + 1)
        plt.figure(figsize=(10, 4), dpi=300)

        plt.subplot(1, 2, 1)
        plt.plot(epochs, train_losses, label='Train Loss', color='blue', lw=3)
        plt.plot(epochs, val_losses, label='Validation Loss', color='orange', lw=3)
        plt.title('Training and Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend(fontsize=10)

        plt.subplot(1, 2, 2)
        plt.plot(epochs, train_accuracies, label='Train Accuracy', color='green', lw=3)
        plt.plot(epochs, val_accuracies, label='Validation Accuracy', color='red', lw=3)
        plt.title('Training and Validation Accuracy')
        plt.xlabel('Epochs')
        plt.ylabel('Accuracy (%)')
        plt.legend(fontsize=10)

        plt.tight_layout()
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1, dpi=300)
        plt.show()

    model_to_evaluate = loras_ct_model.cuda()
    model_to_evaluate_name = 'LoRaS-CT'         
    train_losses_to_plot = comparison_trained_models[model_to_evaluate_name]['train_losses']
    val_losses_to_plot = comparison_trained_models[model_to_evaluate_name]['val_losses']
    train_accs_to_plot = comparison_trained_models[model_to_evaluate_name]['train_accs']
    val_accs_to_plot = comparison_trained_models[model_to_evaluate_name]['val_accs']

    test_predictions, test_labels = get_predictions_and_labels(model_to_evaluate, test_loader)
    plot_confusion_matrix(test_labels, test_predictions, dataset.classes,
                           save_path=f"{img_dir}/confusion_matrix.png")

    test_probs, test_labels_probs = get_predictions_and_labels_probs(model_to_evaluate, test_loader)
    plot_roc_curve(test_labels_probs, test_probs, dataset.classes,
                    save_path=f"{img_dir}/roc_curve.png")

    plot_training_curves(train_losses_to_plot, val_losses_to_plot, train_accs_to_plot, val_accs_to_plot,
                          save_path=f"{img_dir}/training_curves.png")

    print(f"Peak Memory Usage (MB): {comparison_results[model_to_evaluate_name]['Peak Memory (MB)']:.2f}")
    print(f"Per-image Inference Time (ms): {comparison_results[model_to_evaluate_name]['Inference Time (ms)']:.4f}")

    print(f"Number of training images: {len(train_loader.dataset)}")
    print(f"Number of validation images: {len(val_loader.dataset)}")
    print(f"Number of test images: {len(test_loader.dataset)}")

    if hasattr(model_to_evaluate, 'resnet') and hasattr(model_to_evaluate, 'densenet'):
        custom_params = count_custom_parameters(model_to_evaluate, exclude_list=[model_to_evaluate.resnet, model_to_evaluate.densenet])
        print(f'Number of trainable parameters in custom (non-backbone) layers: {custom_params}')



    embed_dim, num_heads, num_layers = 768, GLOBAL_NUM_HEADS, GLOBAL_NUM_LAYERS
    grid_size = 3
    N = grid_size * grid_size        
    rank = GLOBAL_RANK               

    def count_params(module):
        return sum(p.numel() for p in module.parameters() if p.requires_grad)

    attn_factories = {
        'Linformer':               lambda: LinformerAttention(embed_dim, num_heads, seq_len=N),
        'Performer':                lambda: PerformerAttention(embed_dim, num_heads),
        'BigBird':                  lambda: BigBirdAttention(embed_dim, num_heads, block_size=64),
        'Standard MHA (MHA-Net)':   lambda: StandardMHAWrapper(embed_dim, num_heads),
        'LoRa-SMHA (proposed)':     lambda: LowRankSparseMultiheadAttention(embed_dim, num_heads, rank=rank),
    }

    dummy_img = torch.randn(1, 3, 224, 224)
    dummy_seq = torch.randn(1, N, embed_dim)

    table10_rows = {}
    for name, factory in attn_factories.items():
        attn_module = factory()
        attn_params = count_params(attn_module)
        attn_flops, _ = profile(attn_module, inputs=(dummy_seq,), verbose=False)

        full_model = GenericHybridModel(
            num_classes, attn_factory=factory, embed_dim=embed_dim,
            num_layers=num_layers, grid_size=grid_size
        )
        full_params = count_params(full_model)
        full_flops, _ = profile(full_model, inputs=(dummy_img,), verbose=False)

        table10_rows[name] = {
            'Attn. Params': attn_params,
            'Attn. FLOPs': attn_flops,
            'Full-Model Params': full_params,
            'Full-Model FLOPs': full_flops,
        }

    table10 = pd.DataFrame(table10_rows).T[
        ['Attn. Params', 'Attn. FLOPs', 'Full-Model Params', 'Full-Model FLOPs']
    ]

    _table10_name_map = {
        'Standard MHA (MHA-Net)': 'MHA-Net (baseline)',
        'LoRa-SMHA (proposed)': 'LoRaS-CT',
        'Linformer': 'Linformer',
        'Performer': 'Performer',
        'BigBird': 'BigBird',
    }
    table10['Inference Time (ms)'] = [
        isolated_efficiency_table.loc[_table10_name_map[n], 'Inference Time (ms)'] for n in table10.index
    ]
    table10['Peak Memory (MB)'] = [
        isolated_efficiency_table.loc[_table10_name_map[n], 'Peak Memory (MB)'] for n in table10.index
    ]

    baselines = ['Linformer', 'Performer', 'BigBird', 'Standard MHA (MHA-Net)']
    best_attn = table10.loc[baselines, 'Attn. Params'].min()
    best_flops = table10.loc[baselines, 'Attn. FLOPs'].min()
    best_full_p = table10.loc[baselines, 'Full-Model Params'].min()
    best_full_f = table10.loc[baselines, 'Full-Model FLOPs'].min()

    proposed = table10.loc['LoRa-SMHA (proposed)']
    reduction = pd.Series({
        'Attn. Params':       f"{(1 - proposed['Attn. Params'] / best_attn) * 100:.1f}%",
        'Attn. FLOPs':        f"{(1 - proposed['Attn. FLOPs'] / best_flops) * 100:.1f}%",
        'Full-Model Params':  f"{(1 - proposed['Full-Model Params'] / best_full_p) * 100:.1f}%",
        'Full-Model FLOPs':   f"{(1 - proposed['Full-Model FLOPs'] / best_full_f) * 100:.1f}%",
    }, name='Reduction vs. best baseline')

    print(f"Table 10: attention-mechanism efficiency comparison -- {CURRENT_DATASET} (N={N} tokens, embed_dim={embed_dim}, heads={num_heads})\n")
    print(table10.round(0).to_string())
    print()
    print(reduction.to_string())


    print(f"\n{'='*60}\nSection 9c: Trained baseline + SOTA comparison [{CURRENT_DATASET}]\n{'='*60}")

    def _compute_prf(labels, preds):
        return (
            precision_score(labels, preds, average='weighted', zero_division=0),
            recall_score(labels, preds, average='weighted', zero_division=0),
            f1_score(labels, preds, average='weighted', zero_division=0),
        )

    sota_comparison_rows = {}

    _lora_prec, _lora_rec, _lora_f1 = _compute_prf(test_labels, test_predictions)
    sota_comparison_rows['LoRaS-CT'] = {
        'Accuracy (%)': round(comparison_results['LoRaS-CT']['Test Accuracy (%)'], 2),
        'Precision': round(_lora_prec, 4),
        'Recall': round(_lora_rec, 4),
        'F1-score': round(_lora_f1, 4),
    }

    attn_baseline_factories_trained = {
        'Linformer':               lambda: LinformerAttention(embed_dim, num_heads, seq_len=N),
        'Performer':                lambda: PerformerAttention(embed_dim, num_heads),
        'BigBird':                  lambda: BigBirdAttention(embed_dim, num_heads, block_size=64),
        'MHA-Net (Standard MHA)':   lambda: StandardMHAWrapper(embed_dim, num_heads),
    }

    for _name, _factory in attn_baseline_factories_trained.items():
        print(f"\n--- Training attention baseline: {_name} [{CURRENT_DATASET}] ---")

        if _name == 'MHA-Net (Standard MHA)':
            _m = comparison_trained_models['MHA-Net (baseline)']['model'].cuda()
            _acc = comparison_results['MHA-Net (baseline)']['Test Accuracy (%)']
            _preds, _labels = get_predictions_and_labels(_m, test_loader)
            _prec, _rec, _f1 = _compute_prf(_labels, _preds)
            print(f"  Reused from Section 8 -- Acc={_acc:.2f}%")
            sota_comparison_rows[_name] = {
                'Accuracy (%)': round(_acc, 2), 'Precision': round(_prec, 4),
                'Recall': round(_rec, 4), 'F1-score': round(_f1, 4),
            }
            continue

        _m = GenericHybridModel(num_classes, attn_factory=_factory, embed_dim=embed_dim,
                                 num_layers=num_layers, grid_size=grid_size).cuda()
        _crit = DistillationLoss(alpha=0.5, temperature=3.0)
        _opt = optim.SGD(_m.parameters(), lr=0.01, momentum=0.9, weight_decay=_weight_decay)
        train_model_with_distillation(_m, teacher_models, train_loader, val_loader,
                                       _crit, _opt, num_epochs=GLOBAL_NUM_EPOCHS,
                                       restore_best_checkpoint=_restore_best_checkpoint)
        _acc, _ = evaluate_model(_m, test_loader, _crit.ce_loss)
        _preds, _labels = get_predictions_and_labels(_m, test_loader)
        _prec, _rec, _f1 = _compute_prf(_labels, _preds)
        sota_comparison_rows[_name] = {
            'Accuracy (%)': round(_acc, 2), 'Precision': round(_prec, 4),
            'Recall': round(_rec, 4), 'F1-score': round(_f1, 4),
        }
        _m.cpu(); torch.cuda.empty_cache()

    SOTA_BACKBONES = {
        'DeiT B16':  'deit_base_patch16_224',
        'ViT B16':   'vit_base_patch16_224',
        'ViT L16':   'vit_large_patch16_224',
        'ViT L32':   'vit_large_patch32_224',
        'ViT B32':   'vit_base_patch32_224',
        'Swin B4':   'swin_base_patch4_window7_224',
        'Swin V2 S': 'swinv2_small_window16_256',
        'Swin L4':   'swin_large_patch4_window7_224',
    }
    if QUICK_TEST_MODE:
        SOTA_BACKBONES = dict(list(SOTA_BACKBONES.items())[:2])
        print("QUICK_TEST_MODE: only smoke-testing 2/8 SOTA backbones.")

    def _train_plain_classifier(model, train_loader, val_loader, num_epochs, lr=1e-4, input_size=None):
        model = model.cuda()
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
        criterion = nn.CrossEntropyLoss()
        for _epoch in range(num_epochs):
            model.train()
            for images, labels in train_loader:
                images, labels = images.cuda(), labels.cuda()
                if input_size is not None:
                    images = F.interpolate(images, size=input_size, mode='bilinear', align_corners=False)
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                optimizer.step()
        return model

    for _display_name, _timm_name in SOTA_BACKBONES.items():
        print(f"\n--- Fine-tuning SOTA backbone: {_display_name} ({_timm_name}) [{CURRENT_DATASET}] ---")
        try:
            _sota_model = create_model(_timm_name, pretrained=True, num_classes=num_classes)
        except Exception as _e:
            print(f"  Skipping {_display_name}: could not load '{_timm_name}' ({_e})")
            continue
        _backbone_input_size = 256 if _timm_name == 'swinv2_small_window16_256' else None
        try:
            _sota_model = _train_plain_classifier(_sota_model, train_loader, val_loader,
                                                    GLOBAL_NUM_EPOCHS, input_size=_backbone_input_size)
            _preds, _labels = get_predictions_and_labels(_sota_model, test_loader,
                                                            input_size=_backbone_input_size)
            _acc = accuracy_score(_labels, _preds) * 100
            _prec, _rec, _f1 = _compute_prf(_labels, _preds)
            sota_comparison_rows[_display_name] = {
                'Accuracy (%)': round(_acc, 2), 'Precision': round(_prec, 4),
                'Recall': round(_rec, 4), 'F1-score': round(_f1, 4),
            }
        except Exception as _e:
            print(f"  Skipping {_display_name}: {_e}")
        finally:
            _sota_model.cpu(); del _sota_model; torch.cuda.empty_cache()

    sota_comparison_df = pd.DataFrame(sota_comparison_rows).T[
        ['Accuracy (%)', 'Precision', 'Recall', 'F1-score']
    ]
    print(f"\nSection 9c comparison table -- {CURRENT_DATASET} "
          f"(efficient-attention baselines + SOTA backbones vs LoRaS-CT, reused)\n")
    print(sota_comparison_df.to_string())

    _fig, _ax = plt.subplots(figsize=(10, 6))
    _model_order = list(sota_comparison_df.index)
    _ax.plot(_model_order, sota_comparison_df['Precision'], marker='o', label='Precision')
    _ax.plot(_model_order, sota_comparison_df['Recall'], marker='s', linestyle='--', label='Recall')
    _ax.set_xlabel('Models'); _ax.set_ylabel('Score')
    _ax.set_title(f'Precision and Recall across baseline models -- {CURRENT_DATASET}')
    plt.xticks(rotation=45, ha='right')
    _ax.legend(); plt.tight_layout()
    plt.savefig(f"{img_dir}/sota_precision_recall_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()



    def extract_pooled_features(model, data_loader):
        """Extract flattened features + labels from a feature-extractor model."""
        model.eval()
        all_features, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images = images.cuda()
                features = model(images)
                features = features.view(features.size(0), -1)
                all_features.append(features.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.vstack(all_features), np.array(all_labels)


    def plot_tsne(features, labels, title, class_names):
        n_samples = features.shape[0]
        perplexity = min(30, max(n_samples - 1, 1))
        tsne = TSNE(n_components=2, random_state=0, perplexity=perplexity)
        features_2d = tsne.fit_transform(features)

        plt.figure(figsize=(6, 6))
        scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels,
                               cmap=plt.get_cmap('tab10', len(class_names)), alpha=0.7)
        plt.xlabel('t-SNE Component 1')
        plt.ylabel('t-SNE Component 2')
        plt.title(title)
        plt.legend(handles=scatter.legend_elements()[0], labels=class_names, loc='best')
        plt.savefig(f'{img_dir}/{title.lower().replace(" ", "_")}.png', dpi=300)
        plt.show()


    class ResNet18_PooledFeatures(nn.Module):
        def __init__(self):
            super().__init__()
            resnet = models.resnet18(pretrained=True)
            self.model = nn.Sequential(*list(resnet.children())[:-1])  

        def forward(self, x):
            return self.model(x)


    class DenseNet121_PooledFeatures(nn.Module):
        def __init__(self):
            super().__init__()
            densenet = models.densenet121(pretrained=True)
            self.features = densenet.features  

        def forward(self, x):
            x = self.features(x)
            x = F.relu(x, inplace=False)
            x = F.adaptive_avg_pool2d(x, (1, 1))
            return x  


    resnet_pooled_model = ResNet18_PooledFeatures().cuda()
    densenet_pooled_model = DenseNet121_PooledFeatures().cuda()

    resnet_tsne_features, resnet_tsne_labels = extract_pooled_features(resnet_pooled_model, test_loader)
    densenet_tsne_features, densenet_tsne_labels = extract_pooled_features(densenet_pooled_model, test_loader)
    combined_tsne_features = np.concatenate((resnet_tsne_features, densenet_tsne_features), axis=1)

    print("ResNet features shape:", resnet_tsne_features.shape)
    print("DenseNet features shape:", densenet_tsne_features.shape)
    print("Combined features shape:", combined_tsne_features.shape)

    plot_tsne(resnet_tsne_features, resnet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of ResNet18 Features', dataset.classes)
    plot_tsne(densenet_tsne_features, densenet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of DenseNet121 Features', dataset.classes)
    plot_tsne(combined_tsne_features, resnet_tsne_labels, f'{CURRENT_DATASET} - t-SNE Plot of Combined Features', dataset.classes)

    def extract_hybrid_features_for_tsne(model, data_loader):
        model.eval()
        all_features, all_labels = [], []
        with torch.no_grad():
            for images, labels in data_loader:
                images = images.cuda()
                _, feats = model(images, return_features=True)
                feats = feats.view(feats.size(0), -1)
                all_features.append(feats.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return np.vstack(all_features), np.array(all_labels)


    hybrid_tsne_features, hybrid_tsne_labels = extract_hybrid_features_for_tsne(loras_ct_model, test_loader)

    n_samples = hybrid_tsne_features.shape[0]
    perplexity = min(30, max(n_samples - 1, 1))
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    hybrid_tsne_2d = tsne.fit_transform(hybrid_tsne_features)

    df_tsne = pd.DataFrame(data=hybrid_tsne_2d, columns=['Dim1', 'Dim2'])
    df_tsne['Label'] = hybrid_tsne_labels

    plt.figure(figsize=(10, 8))
    sns.scatterplot(x='Dim1', y='Dim2', hue='Label', data=df_tsne, palette='tab10', marker='o', alpha=0.7)
    plt.title(f'{CURRENT_DATASET} - t-SNE Visualization of Hybrid Model Features')
    plt.xlabel('Dimension 1')
    plt.ylabel('Dimension 2')
    plt.legend(title='Classes', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.savefig(f'{img_dir}/hybrid_tsne.png', dpi=300, bbox_inches='tight')
    plt.show()



    from lime import lime_image
    from skimage.segmentation import mark_boundaries  

    def lime_predict_fn(images):
        images_tensor = torch.tensor(images).permute(0, 3, 1, 2).float().cuda()
        with torch.no_grad():
            outputs = model_to_evaluate(images_tensor)
        return F.softmax(outputs, dim=1).cpu().numpy()

    lime_explainer = lime_image.LimeImageExplainer()

    lime_image_idx = 0
    lime_test_image = test_loader.dataset[lime_image_idx][0].unsqueeze(0)
    lime_test_image_np = lime_test_image.squeeze(0).cpu().numpy().transpose(1, 2, 0)  

    lime_explanation = lime_explainer.explain_instance(
        lime_test_image_np,
        lime_predict_fn,
        top_labels=5,
        hide_color=0,
        num_samples=1000,
    )

    lime_label_to_explain = lime_explanation.top_labels[0]
    lime_temp, lime_mask = lime_explanation.get_image_and_mask(
        lime_label_to_explain, positive_only=True, num_features=10, hide_rest=False
    )

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(lime_test_image_np)
    plt.title(f"{CURRENT_DATASET} - Original Image")

    plt.subplot(1, 2, 2)
    plt.imshow(mark_boundaries(lime_temp, lime_mask))
    plt.title(f"{CURRENT_DATASET} - LIME Explanation for Class: {lime_label_to_explain}")
    plt.show()




    import shap

    shap_image_idx = 0
    shap_test_image, _ = test_loader.dataset[shap_image_idx]
    shap_test_image = shap_test_image.unsqueeze(0)
    print("Shape of the image tensor before conversion:", shap_test_image.shape)

    shap_n_background = min(50, len(test_loader.dataset))
    shap_background = torch.stack([test_loader.dataset[i][0] for i in range(shap_n_background)]).cuda()

    model_to_evaluate.eval()  

    shap_explainer = shap.GradientExplainer(model_to_evaluate, shap_background)

    shap_values, indexes = shap_explainer.shap_values(
        shap_test_image.cuda(), ranked_outputs=1
    )
    shap_values_class = shap_values[0]  

    shap_map = np.mean(np.abs(shap_values_class), axis=0)
    shap_map = (shap_map - shap_map.min()) / (shap_map.max() - shap_map.min() + 1e-8)
    if shap_map.ndim == 3:
        shap_map = shap_map.mean(axis=0)

    shap_display_image = shap_test_image.squeeze(0).permute(1, 2, 0).cpu().numpy()
    shap_display_image = np.clip(shap_display_image, 0, 1)

    predicted_class_idx = int(indexes[0][0])
    predicted_class_name = (
        dataset.classes[predicted_class_idx] if "dataset" in globals() else predicted_class_idx
    )

    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(shap_display_image)
    plt.axis('off')
    plt.title(f'{CURRENT_DATASET} - Original Image')

    plt.subplot(1, 2, 2)
    plt.imshow(shap_display_image)
    plt.imshow(shap_map, cmap='jet', alpha=0.5)
    plt.axis('off')
    plt.title(f'SHAP Explanation (class: {predicted_class_name})')
    plt.show()




    def compute_metrics_table(models_dict, data_loader, comparison_results):
        rows = {}
        for name, model in models_dict.items():
            preds, labels = get_predictions_and_labels(model, data_loader)
            rows[name] = {
                'Accuracy':  comparison_results[name]['Test Accuracy (%)'] / 100,  
                'Precision': precision_score(labels, preds, average='weighted', zero_division=0),
                'Recall':    recall_score(labels, preds, average='weighted', zero_division=0),
                'F1-score':  f1_score(labels, preds, average='weighted', zero_division=0),
            }
        df = pd.DataFrame(rows).T[['Accuracy', 'Precision', 'Recall', 'F1-score']]
        return df.round(4)

    models_dict = {
        'LoRaS-CT': loras_ct_model,
        'MHA-Net (baseline)': mha_net_model,
    }

    metrics_table = compute_metrics_table(models_dict, test_loader, comparison_results)
    print(f"Table 3: Performance metrics on {CURRENT_DATASET} (Accuracy matches Section 8 FINAL COMPARISON exactly)\n")
    print(metrics_table.to_string())

    
    for _model_name, _row in metrics_table.iterrows():
        save_comparison_metrics_to_checkpoint(
            CURRENT_DATASET,
            _model_name,
            _row.to_dict()
        )

    efficiency_table = pd.DataFrame(comparison_results).T[
        ['Params (M)', 'FLOPs (G)', 'Inference Time (ms)', 'Peak Memory (MB)', 'Test Accuracy (%)']
    ].round(4)
    print(f"Table 5: Computational efficiency on {CURRENT_DATASET} (identical to Section 8 FINAL COMPARISON)\n")
    print(efficiency_table.to_string())

    def count_all_parameters(module):
        return sum(p.numel() for p in module.parameters())

    def param_breakdown(models_dict):
        rows = {}
        backbone_params = None
        for name, model in models_dict.items():
            total_trainable = count_parameters(model)  
            if hasattr(model, 'resnet') and hasattr(model, 'densenet'):
                head_only = count_custom_parameters(model, exclude_list=[model.resnet, model.densenet])
                if backbone_params is None:
                    backbone_params = count_all_parameters(model.resnet) + count_all_parameters(model.densenet)
            else:
                head_only = total_trainable
            rows[name] = {
                'Total trainable params': f'{total_trainable:,}',
                'Transformer+head only (excl. CNN backbones)': f'{head_only:,}',
            }
        df = pd.DataFrame(rows).T
        return df, backbone_params

    param_table, shared_backbone_params = param_breakdown(models_dict)
    print(param_table.to_string())
    print(f"\nShared CNN backbone params (ResNet18+DenseNet121 features, same for both models): {shared_backbone_params:,}")

    print(f"\n[{CURRENT_DATASET}] Consistency check vs. Section 8 FINAL COMPARISON (Params (M)):")
    for name in models_dict:
        total_trainable = count_parameters(models_dict[name])
        reported = comparison_results[name]['Params (M)'] * 1e6
        status = "OK" if abs(total_trainable - reported) < 1 else "MISMATCH - investigate"
        print(f"  {name}: param_breakdown={total_trainable:,}  vs  comparison_results={reported:,.0f}  [{status}]")



    model_builders = {
        'LoRaS-CT': lambda: HybridStudentModel(num_classes, grid_size=3).cuda(),
        'MHA-Net (baseline)': lambda: MHANetBaseline(num_classes, grid_size=3).cuda(),
    }

    multiseed_results = {name: {'accuracy': [], 'f1': [], 'precision': [], 'recall': []} for name in model_builders}

    for seed in SEED_LIST:
        print(f"\n{'#'*70}\n# SEED {seed}\n{'#'*70}")
        set_all_seeds(seed)
        if CURRENT_DATASET == "LC25000":
            _train_ds, _val_ds, _test_ds, _, _ = build_lc25000_grouped_split(
                DATASET_CFG, train_val_transform, test_transform, seed=seed
            )
            _train_ds = quick_subset(_train_ds, QUICK_TEST_MAX_TRAIN)
            _val_ds = quick_subset(_val_ds, QUICK_TEST_MAX_VAL)
            _test_ds = quick_subset(_test_ds, QUICK_TEST_MAX_TEST)
            seed_train_loader = DataLoader(_train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
            seed_val_loader = DataLoader(_val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
            seed_test_loader = DataLoader(_test_ds, batch_size=batch_size, shuffle=False, num_workers=2)
        else:
            seed_train_loader, seed_val_loader, seed_test_loader = make_split(dataset_path, seed, use_class_balancing=_use_class_balancing)

        for name, builder in model_builders.items():
            set_all_seeds(seed)  
            model = builder()
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=_weight_decay)

            model, *_ = train_model_with_distillation(
                model, teacher_models, seed_train_loader, seed_val_loader,
                criterion, optimizer, num_epochs=STAT_NUM_EPOCHS,
                restore_best_checkpoint=_restore_best_checkpoint
            )

            acc, _, y_true, y_pred = evaluate_model(model, seed_test_loader, criterion.ce_loss, return_predictions=True)
            f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
            prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
            rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)

            multiseed_results[name]['accuracy'].append(acc)
            multiseed_results[name]['f1'].append(f1)
            multiseed_results[name]['precision'].append(prec)
            multiseed_results[name]['recall'].append(rec)

            print(f"[seed {seed}] {name}: Acc={acc:.2f}%  F1={f1:.4f}")

            del model
            torch.cuda.empty_cache()
            gc.collect()

    summary_rows = {}
    for name, m in multiseed_results.items():
        summary_rows[name] = {
            'Accuracy (%)': f"{np.mean(m['accuracy']):.2f} +/- {np.std(m['accuracy']):.2f}",
            'F1-score':     f"{np.mean(m['f1']):.4f} +/- {np.std(m['f1']):.4f}",
            'Precision':    f"{np.mean(m['precision']):.4f} +/- {np.std(m['precision']):.4f}",
            'Recall':       f"{np.mean(m['recall']):.4f} +/- {np.std(m['recall']):.4f}",
            'N runs':       len(m['accuracy']),
        }
    stat_summary_df = pd.DataFrame(summary_rows).T
    print(f"Table: Mean +/- SD over {N_SEEDS} independent seeds -- {CURRENT_DATASET}\n")
    print(stat_summary_df.to_string())

    acc_a = multiseed_results['LoRaS-CT']['accuracy']
    acc_b = multiseed_results['MHA-Net (baseline)']['accuracy']
    if N_SEEDS < 2:
        print(f"\nPaired t-test skipped: N_SEEDS={N_SEEDS} (need >= 2 seeds to estimate variance).")
        print("  This is expected under QUICK_TEST_MODE. Set QUICK_TEST_MODE = False for a real test.")
    else:
        t_stat, p_value = stats.ttest_rel(acc_a, acc_b)
        print(f"\nPaired t-test (LoRaS-CT vs MHA-Net, accuracy, N={N_SEEDS} seeds):")
        print(f"  t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")
        print(f"  {'Statistically significant at alpha=0.05' if p_value < 0.05 else 'NOT statistically significant at alpha=0.05'}")
        print("  NOTE: with N_SEEDS=3 this test has very low statistical power - treat the p-value")
        print("  as indicative only. Increase N_SEEDS to 5-10.")

    fig, ax = plt.subplots(figsize=(6, 4))
    names = list(multiseed_results.keys())
    means = [np.mean(multiseed_results[n]['accuracy']) for n in names]
    stds = [np.std(multiseed_results[n]['accuracy']) for n in names]
    colors = ['#2E86AB', '#A23B72']
    ax.bar(names, means, yerr=stds, capsize=8, color=colors[:len(names)])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Test Accuracy Mean +/- SD over {N_SEEDS} Seeds')
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.text(i, m + s + 0.3, f'{m:.2f}+/-{s:.2f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/stat_significance_barplot.png', dpi=150)
    plt.show()



    gradcam_target_layer = model_to_evaluate.resnet.features[-1]
    gradcam = GradCAM(model_to_evaluate, gradcam_target_layer)

    n_examples = 4
    fig, axes = plt.subplots(2, n_examples, figsize=(4 * n_examples, 8))

    for i in range(n_examples):
        img_tensor, true_label = test_loader.dataset[i]
        input_tensor = img_tensor.unsqueeze(0).cuda()

        cam, pred_class = gradcam.generate(input_tensor)

        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

        true_name = dataset.classes[true_label] if "dataset" in globals() else true_label
        pred_name = dataset.classes[pred_class] if "dataset" in globals() else pred_class

        axes[0, i].imshow(img_np)
        axes[0, i].axis('off')
        axes[0, i].set_title(f"True: {true_name}", fontsize=9)

        axes[1, i].imshow(img_np)
        axes[1, i].imshow(cam, cmap='jet', alpha=0.5)
        axes[1, i].axis('off')
        axes[1, i].set_title(f"Grad-CAM (pred: {pred_name})", fontsize=9)

    plt.suptitle(f"Grad-CAM explanations -- {CURRENT_DATASET} ({model_to_evaluate_name})", y=1.02)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/gradcam_examples.png', dpi=150, bbox_inches='tight')
    plt.show()

    gradcam.remove()



    ABLATION_EPOCHS = GLOBAL_NUM_EPOCHS  

    def build_full_model():
        return HybridStudentModel(num_classes, grid_size=3).cuda()

    def build_no_lowrank():
        return GenericHybridModel(
            num_classes, attn_factory=lambda: FullRankSparseAttention(768, GLOBAL_NUM_HEADS, sparsity_ratio=0.5), grid_size=3
        ).cuda()

    def build_no_sparsity():
        return GenericHybridModel(
            num_classes, attn_factory=lambda: LowRankSparseMultiheadAttention(768, GLOBAL_NUM_HEADS, rank=GLOBAL_RANK, sparsity_ratio=1.0),
            grid_size=3
        ).cuda()

    def build_resnet_only():
        return SingleBackboneHybridModel(num_classes, backbone='resnet', grid_size=3).cuda()

    def build_densenet_only():
        return SingleBackboneHybridModel(num_classes, backbone='densenet', grid_size=3).cuda()

    ablation_configs = {
        'Full LoRaS-CT (all components)':    (build_full_model,    True),
        'w/o Low-Rank Factorization':        (build_no_lowrank,    True),
        'w/o Sparsity (top-k masking)':      (build_no_sparsity,   True),
        'w/o Knowledge Distillation':        (build_full_model,    False),
        'w/o DenseNet branch (ResNet only)': (build_resnet_only,   True),
        'w/o ResNet branch (DenseNet only)': (build_densenet_only, True),
    }

    ablation_results = {}
    for name, (builder, use_kd) in ablation_configs.items():
        print(f"\n{'='*60}\nAblation: {name}\n{'='*60}")

        if name == 'Full LoRaS-CT (all components)':
            eval_model = comparison_trained_models['LoRaS-CT']['model'].cuda()
            acc = comparison_results['LoRaS-CT']['Test Accuracy (%)']
            params = comparison_results['LoRaS-CT']['Params (M)'] * 1e6
            _, _, y_true, y_pred = evaluate_model(eval_model, test_loader, nn.CrossEntropyLoss(), return_predictions=True)
            f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
            print(f"[{name}] Reused from Section 8 -- Acc={acc:.2f}%  F1={f1:.4f}  Params={params/1e6:.2f}M")
            ablation_results[name] = {'Accuracy (%)': acc, 'F1-score': f1, 'Params (M)': params / 1e6}
            continue

        set_all_seeds(42)
        model = builder()
        eval_criterion = nn.CrossEntropyLoss()
        if use_kd:
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=_weight_decay)
            model, *_ = train_model_with_distillation(
                model, teacher_models, train_loader, val_loader, criterion, optimizer, num_epochs=ABLATION_EPOCHS,
                restore_best_checkpoint=_restore_best_checkpoint
            )
            eval_criterion = criterion.ce_loss
        else:
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=_weight_decay)
            train_model_plain(model, train_loader, val_loader, eval_criterion, optimizer, num_epochs=ABLATION_EPOCHS)

        acc, _, y_true, y_pred = evaluate_model(model, test_loader, eval_criterion, return_predictions=True)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        params = count_parameters(model)

        ablation_results[name] = {'Accuracy (%)': acc, 'F1-score': f1, 'Params (M)': params / 1e6}
        print(f"[{name}] Acc={acc:.2f}%  F1={f1:.4f}  Params={params/1e6:.2f}M")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    ablation_df = pd.DataFrame(ablation_results).T[['Accuracy (%)', 'F1-score', 'Params (M)']]
    full_acc = ablation_df.loc['Full LoRaS-CT (all components)', 'Accuracy (%)']
    ablation_df['Delta Accuracy (pp)'] = (ablation_df['Accuracy (%)'] - full_acc).round(2)
    print(f"Table: Component-wise ablation study -- {CURRENT_DATASET}\n")
    print(ablation_df.round(4).to_string())

    fig, ax = plt.subplots(figsize=(9, 5))
    names = list(ablation_results.keys())
    accs = [ablation_results[n]['Accuracy (%)'] for n in names]
    colors = ['#2E86AB'] + ['#A23B72'] * (len(names) - 1)
    bars = ax.barh(names, accs, color=colors)
    ax.axvline(full_acc, color='gray', linestyle='--', label='Full model accuracy')
    ax.set_xlabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Component-wise Ablation Study')
    ax.legend()
    for bar, acc in zip(bars, accs):
        ax.text(acc + 0.3, bar.get_y() + bar.get_height() / 2, f'{acc:.2f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{img_dir}/ablation_study.png', dpi=150)
    plt.show()



    foundation_models_to_test = [] if QUICK_TEST_MODE else ['UNI', 'CONCH', 'Virchow']  
    if QUICK_TEST_MODE:
        print("QUICK_TEST_MODE: skipping foundation-model downloads (multi-GB gated weights).")
        print("Set QUICK_TEST_MODE = False in Cell 5 to run this section for real.")
    elif not globals().get('hf_login_ok', False):
        print("Hugging Face login was not successful (see the cell above) - skipping all three")
        print("gated foundation models rather than hitting the same 401 three times.")
        foundation_models_to_test = []
    foundation_results = {}

    for fm_name in foundation_models_to_test:
        print(f"\n{'='*60}\nFoundation model: {fm_name}\n{'='*60}")
        try:
            encoder, embed_dim = load_foundation_encoder(fm_name)
            model = LinearProbeHead(encoder, embed_dim, num_classes).cuda()

            optimizer = optim.Adam(model.head.parameters(), lr=1e-3, weight_decay=0.0)  
            criterion = nn.CrossEntropyLoss()
            train_model_plain(model, train_loader, val_loader, criterion, optimizer,
                               num_epochs=FOUNDATION_MODEL_EPOCHS)

            acc, _, y_true, y_pred = evaluate_model(model, test_loader, criterion, return_predictions=True)
            f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
            params = count_parameters(model)  

            dummy_input = torch.randn(1, 3, 224, 224).cuda()
            inf_time = measure_inference_time(model, dummy_input)
            memory = measure_memory(model, dummy_input)

            foundation_results[fm_name] = {
                'Test Accuracy (%)': acc, 'F1-score': f1,
                'Trainable Params (M)': params / 1e6,
                'Total Params (M)': sum(p.numel() for p in model.parameters()) / 1e6,
                'Inference Time (ms)': inf_time, 'Peak Memory (MB)': memory,
            }
            print(f"[{fm_name}] Acc={acc:.2f}%  F1={f1:.4f}")

            del model, encoder
            torch.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            print(f"[{fm_name}] SKIPPED -- {type(e).__name__}: {e}")
            print("  (Needs internet + HF gated-license access in this environment. Run "
                  "`huggingface-cli login` after accepting the license on the model's HF page.)")
            foundation_results[fm_name] = {
                'Test Accuracy (%)': np.nan, 'F1-score': np.nan,
                'Trainable Params (M)': np.nan, 'Total Params (M)': np.nan,
                'Inference Time (ms)': np.nan, 'Peak Memory (MB)': np.nan,
            }

    combined_results = dict(foundation_results)
    combined_results['LoRaS-CT (ours)'] = {
        'Test Accuracy (%)': ablation_results['Full LoRaS-CT (all components)']['Accuracy (%)']
            if 'ablation_results' in globals() else np.nan,
        'F1-score': ablation_results['Full LoRaS-CT (all components)']['F1-score']
            if 'ablation_results' in globals() else np.nan,
        'Trainable Params (M)': count_parameters(loras_ct_model) / 1e6,
        'Total Params (M)': sum(p.numel() for p in loras_ct_model.parameters()) / 1e6,
        'Inference Time (ms)': np.nan, 'Peak Memory (MB)': np.nan,
    }

    foundation_comparison_df = pd.DataFrame(combined_results).T
    print(f"Table: LoRaS-CT vs pathology foundation models (linear probe) -- {CURRENT_DATASET}\n")
    print(foundation_comparison_df.round(4).to_string())

    fig, ax = plt.subplots(figsize=(8, 5))
    valid = foundation_comparison_df.dropna(subset=['Test Accuracy (%)'])
    bar_colors = ['#F18F01'] * (len(valid) - 1) + ['#2E86AB'] if 'LoRaS-CT (ours)' in valid.index else ['#F18F01'] * len(valid)
    ax.bar(valid.index, valid['Test Accuracy (%)'], color=bar_colors)
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'LoRaS-CT vs Pathology Foundation Models (linear probe, {CURRENT_DATASET})')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/foundation_model_comparison.png', dpi=150)
    plt.show()



    from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold
    from scipy import stats as scipy_stats

    K_FOLDS = 2 if QUICK_TEST_MODE else 5
    KFOLD_NUM_EPOCHS = GLOBAL_NUM_EPOCHS  

    if CURRENT_DATASET == "LC25000":
        lc25000_all_paths, kfold_labels, kfold_groups, _lc_classes = get_lc25000_pooled_samples(DATASET_CFG)
        all_indices = np.arange(len(lc25000_all_paths))
    else:
        full_ds_for_kfold = datasets.ImageFolder(dataset_path)
        all_indices = np.arange(len(full_ds_for_kfold))
        kfold_groups = get_patient_groups(full_ds_for_kfold) if _use_class_balancing else None
        kfold_labels = np.asarray(full_ds_for_kfold.targets)

    if kfold_groups is not None:
        _group_label = "original-image groups" if CURRENT_DATASET == "LC25000" else "Patient/slide IDs"
        print(f"[{CURRENT_DATASET}] {_group_label} detected - using the 500-restart "
              f"balanced group-fold assignment (group-disjoint AND class-balanced).")
        kfold_assignment = _balanced_group_folds(kfold_labels, kfold_groups, K_FOLDS, seed=42, n_restarts=500)
        for _f in range(K_FOLDS):
            _mask = kfold_assignment == _f
            _counts = np.bincount(kfold_labels[_mask], minlength=2)
            print(f"  Fold {_f + 1}: images={int(_mask.sum())}, groups={len(set(kfold_groups[_mask]))}, "
                  f"classes={_counts.tolist()}, ratio={(_counts / max(_counts.sum(),1)).round(3).tolist()}")
        fold_iter = ((all_indices[kfold_assignment != _f], all_indices[kfold_assignment == _f]) for _f in range(K_FOLDS))
    else:
        kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
        fold_iter = kf.split(all_indices)

    kfold_results = {name: [] for name in model_builders}

    for fold_idx, (trainval_idx, test_idx) in enumerate(fold_iter):
        print(f"\n{'#'*70}\n# FOLD {fold_idx + 1}/{K_FOLDS}\n{'#'*70}")
        set_all_seeds(42 + fold_idx)

        if kfold_groups is not None:
            trainval_groups = kfold_groups[trainval_idx]
            trainval_labels = kfold_labels[trainval_idx]
            n_val_splits = max(2, round(1 / 0.1))
            tr_sub, val_sub, _ = _split_from_balanced_groups(
                trainval_labels, trainval_groups, n_val_splits, seed=1000 + fold_idx, heldout_fold=0
            )
            train_idx, val_idx = trainval_idx[tr_sub], trainval_idx[val_sub]
        else:
            n_val = int(len(trainval_idx) * 0.1)
            val_idx = trainval_idx[:n_val]
            train_idx = trainval_idx[n_val:]

        if CURRENT_DATASET == "LC25000":
            train_subset = PathListDataset(
                list(zip(lc25000_all_paths[train_idx].tolist(), kfold_labels[train_idx].tolist())),
                transform=train_val_transform,
            )
            val_subset = PathListDataset(
                list(zip(lc25000_all_paths[val_idx].tolist(), kfold_labels[val_idx].tolist())),
                transform=train_val_transform,
            )
            test_subset = PathListDataset(
                list(zip(lc25000_all_paths[test_idx].tolist(), kfold_labels[test_idx].tolist())),
                transform=test_transform,
            )
        else:
            train_subset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), train_idx)
            val_subset = Subset(datasets.ImageFolder(dataset_path, transform=train_val_transform), val_idx)
            test_subset = Subset(datasets.ImageFolder(dataset_path, transform=test_transform), test_idx)
        train_subset = quick_subset(train_subset, QUICK_TEST_MAX_TRAIN)
        val_subset = quick_subset(val_subset, QUICK_TEST_MAX_VAL)
        test_subset = quick_subset(test_subset, QUICK_TEST_MAX_TEST)

        fold_train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
        fold_val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
        fold_test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=2)

        for name, builder in model_builders.items():
            set_all_seeds(42 + fold_idx)
            model = builder()
            criterion = DistillationLoss(alpha=0.5, temperature=3.0)
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=_weight_decay)
            model, *_ = train_model_with_distillation(
                model, teacher_models, fold_train_loader, fold_val_loader, criterion, optimizer,
                num_epochs=KFOLD_NUM_EPOCHS, restore_best_checkpoint=_restore_best_checkpoint
            )
            acc, _ = evaluate_model(model, fold_test_loader, criterion.ce_loss)

            kfold_results[name].append(acc)
            print(f"[Fold {fold_idx + 1}] {name}: Test Acc = {acc:.2f}%")

            del model
            torch.cuda.empty_cache()
            gc.collect()

    def confidence_interval_95(values):
        values = np.array(values)
        mean = values.mean()
        if len(values) > 1:
            sem = scipy_stats.sem(values)
            ci = scipy_stats.t.interval(0.95, df=len(values) - 1, loc=mean, scale=sem)
        else:
            ci = (mean, mean)
        return mean, values.std(), ci

    kfold_summary_rows = {}
    for name, accs in kfold_results.items():
        mean, std, ci = confidence_interval_95(accs)
        kfold_summary_rows[name] = {
            'Mean Accuracy (%)': round(mean, 2),
            'Std Dev': round(std, 2),
            '95% CI Lower': round(ci[0], 2),
            '95% CI Upper': round(ci[1], 2),
            'K': len(accs),
        }
    kfold_summary_df = pd.DataFrame(kfold_summary_rows).T
    print(f"Table: {K_FOLDS}-Fold Cross-Validation Results with 95% Confidence Intervals - {CURRENT_DATASET}\n")
    print(kfold_summary_df.to_string())

    if K_FOLDS < 2:
        print(f"\nPaired t-test skipped: K_FOLDS={K_FOLDS} (need >= 2 folds to estimate variance).")
    else:
        t_stat, p_value = scipy_stats.ttest_rel(kfold_results['LoRaS-CT'], kfold_results['MHA-Net (baseline)'])
        print(f"\nPaired t-test across {K_FOLDS} folds (LoRaS-CT vs MHA-Net): t={t_stat:.4f}, p={p_value:.4f}")
        print(f"  {'Statistically significant at alpha=0.05' if p_value < 0.05 else 'NOT statistically significant at alpha=0.05'}")

    fig, ax = plt.subplots(figsize=(6, 4))
    names = list(kfold_results.keys())
    means = [np.mean(kfold_results[n]) for n in names]
    cis = [confidence_interval_95(kfold_results[n])[2] for n in names]
    errs = np.array([[m - ci[0], ci[1] - m] for m, ci in zip(means, cis)]).T
    ax.bar(names, means, yerr=errs, capsize=8, color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: {K_FOLDS}-Fold CV Mean Accuracy with 95% CI')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/kfold_cv_results.png', dpi=150)
    plt.show()



    HPARAM_ABLATION_EPOCHS = GLOBAL_NUM_EPOCHS  

    layer_configs = [1, 2] if QUICK_TEST_MODE else [1, 2, 3, 4]
    head_configs = [4, 8] if QUICK_TEST_MODE else [2, 4, 8, 16]  

    hparam_results_layers = {}
    for nl in layer_configs:
        print(f"\n--- num_layers = {nl} ---")

        if nl == GLOBAL_NUM_LAYERS:
            acc = comparison_results['LoRaS-CT']['Test Accuracy (%)']
            params = comparison_results['LoRaS-CT']['Params (M)'] * 1e6
            print(f"[num_layers={nl}] Reused from Section 8 -- Acc={acc:.2f}%  Params={params/1e6:.2f}M")
            hparam_results_layers[nl] = {'Test Accuracy (%)': acc, 'Params (M)': params / 1e6}
            continue

        set_all_seeds(42)
        model = HybridStudentModel(num_classes, num_layers=nl, grid_size=3).cuda()
        criterion = DistillationLoss(alpha=0.5, temperature=3.0)
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=_weight_decay)
        model, *_ = train_model_with_distillation(
            model, teacher_models, train_loader, val_loader, criterion, optimizer,
            num_epochs=HPARAM_ABLATION_EPOCHS, restore_best_checkpoint=_restore_best_checkpoint
        )
        acc, _ = evaluate_model(model, test_loader, criterion.ce_loss)
        params = count_parameters(model)
        hparam_results_layers[nl] = {'Test Accuracy (%)': acc, 'Params (M)': params / 1e6}
        print(f"[num_layers={nl}] Acc={acc:.2f}%  Params={params/1e6:.2f}M")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    hparam_results_heads = {}
    for nh in head_configs:
        print(f"\n- num_heads = {nh} -")

        if nh == GLOBAL_NUM_HEADS:
            acc = comparison_results['LoRaS-CT']['Test Accuracy (%)']
            print(f"[num_heads={nh}] Reused from Section 8 - Acc={acc:.2f}%")
            hparam_results_heads[nh] = {'Test Accuracy (%)': acc}
            continue

        set_all_seeds(42)
        model = HybridStudentModel(num_classes, num_heads=nh, grid_size=3).cuda()
        criterion = DistillationLoss(alpha=0.5, temperature=3.0)
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=_weight_decay)
        model, *_ = train_model_with_distillation(
            model, teacher_models, train_loader, val_loader, criterion, optimizer,
            num_epochs=HPARAM_ABLATION_EPOCHS, restore_best_checkpoint=_restore_best_checkpoint
        )
        acc, _ = evaluate_model(model, test_loader, criterion.ce_loss)
        hparam_results_heads[nh] = {'Test Accuracy (%)': acc}
        print(f"[num_heads={nh}] Acc={acc:.2f}%")

        del model
        torch.cuda.empty_cache()
        gc.collect()

    layers_df = pd.DataFrame(hparam_results_layers).T
    layers_df.index.name = 'num_layers'
    print(f"Table: Ablation over number of transformer layers -- {CURRENT_DATASET}\n")
    print(layers_df.round(4).to_string())

    heads_df = pd.DataFrame(hparam_results_heads).T
    heads_df.index.name = 'num_heads'
    print(f"\nTable: Ablation over number of attention heads -- {CURRENT_DATASET}\n")
    print(heads_df.round(4).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].plot(list(hparam_results_layers.keys()),
                 [v['Test Accuracy (%)'] for v in hparam_results_layers.values()],
                 marker='o', color='#2E86AB')
    axes[0].set_xlabel('Number of Transformer Layers')
    axes[0].set_ylabel('Test Accuracy (%)')
    axes[0].set_title(f'{CURRENT_DATASET}: Accuracy vs. Number of Layers')
    axes[0].set_xticks(layer_configs)

    axes[1].plot(list(hparam_results_heads.keys()),
                 [v['Test Accuracy (%)'] for v in hparam_results_heads.values()],
                 marker='s', color='#A23B72')
    axes[1].set_xlabel('Number of Attention Heads')
    axes[1].set_ylabel('Test Accuracy (%)')
    axes[1].set_title(f'{CURRENT_DATASET}: Accuracy vs. Number of Attention Heads')
    axes[1].set_xticks(head_configs)

    plt.tight_layout()
    plt.savefig(f'{img_dir}/hparam_ablation_layers_heads.png', dpi=150)
    plt.show()

    print("\nNote: the KD on/off ablation for this is already reported")
    print("in Section 17 ('w/o Knowledge Distillation' row of the component-wise ablation table).")



    
    TEACHER_FT_EPOCHS = GLOBAL_NUM_EPOCHS
    TEACHER_LP_EPOCHS = GLOBAL_NUM_EPOCHS
    teacher_names = ['ViT-B/16', 'DeiT-B/16', 'Swin-B/4']
    teacher_archs = [
        'vit_base_patch16_224',
        'deit_base_patch16_224',
        'swin_base_patch4_window7_224',
    ]

    def evaluate_teacher_standalone(teacher, data_loader):
        acc, _ = evaluate_model(teacher, data_loader, nn.CrossEntropyLoss())
        return acc

    def build_fresh_teacher(arch_name, num_classes):
        return create_model(arch_name, pretrained=True, num_classes=num_classes).cuda()

   
    print(f"[{CURRENT_DATASET}] Teacher accuracy: RAW PRETRAINED (no dataset-specific training)")
    print("(ImageNet-pretrained backbone, but the num_classes-sized classification head")
    print(" has never been trained on this dataset - expect close to chance level):\n")

    raw_pretrained_accs = {}
    for name, teacher in zip(teacher_names, teacher_models):
        acc = evaluate_teacher_standalone(teacher, test_loader)
        raw_pretrained_accs[name] = acc
        print(f"  {name}: {acc:.2f}%")

    # ------------------------------------------------------------
    # 2. LINEAR PROBING
    
    print(f"\n[{CURRENT_DATASET}] Linear probing each teacher (fresh pretrained teacher; frozen backbone)...\n")
    linear_probe_accs = {}

    for name, arch_name in zip(teacher_names, teacher_archs):
        print(f"--- {name}: Linear Probe ---")
        teacher_lp = build_fresh_teacher(arch_name, num_classes)

        for p in teacher_lp.parameters():
            p.requires_grad = False

        classifier = teacher_lp.get_classifier()
        for p in classifier.parameters():
            p.requires_grad = True

        optimizer = optim.SGD(
            filter(lambda p: p.requires_grad, teacher_lp.parameters()),
            lr=1e-3, momentum=0.9, weight_decay=_weight_decay
        )
        criterion = nn.CrossEntropyLoss()

        train_model_plain(
            teacher_lp, train_loader, val_loader, criterion, optimizer,
            num_epochs=TEACHER_LP_EPOCHS
        )

        acc = evaluate_teacher_standalone(teacher_lp, test_loader)
        linear_probe_accs[name] = acc
        print(f"  {name} Linear Probe Test Accuracy: {acc:.2f}%\n")

        del teacher_lp
        torch.cuda.empty_cache()
        gc.collect()

    # ------------------------------------------------------------
    # 3. FULL FINE-TUNING
    # ------------------------------------------------------------
    print(f"[{CURRENT_DATASET}] Full fine-tuning each teacher (fresh pretrained teacher; all parameters trainable)...\n")
    full_ft_accs = {}

    for name, arch_name in zip(teacher_names, teacher_archs):
        print(f"--- {name}: Full Fine-Tuning ---")
        teacher_ft = build_fresh_teacher(arch_name, num_classes)

        optimizer = optim.SGD(
            teacher_ft.parameters(), lr=1e-4, momentum=0.9, weight_decay=_weight_decay
        )
        criterion = nn.CrossEntropyLoss()

        train_model_plain(
            teacher_ft, train_loader, val_loader, criterion, optimizer,
            num_epochs=TEACHER_FT_EPOCHS
        )

        acc = evaluate_teacher_standalone(teacher_ft, test_loader)
        full_ft_accs[name] = acc
        print(f"  {name} Fine-Tuned Test Accuracy: {acc:.2f}%\n")

        del teacher_ft
        torch.cuda.empty_cache()
        gc.collect()

    teacher_comparison_df = pd.DataFrame({
        'Raw pretrained (%)': raw_pretrained_accs,
        'Linear probe (%)': linear_probe_accs,
        'Fine-tuned (%)': full_ft_accs,
    })

    print(f"\nTable: Teacher standalone performance -- {CURRENT_DATASET}\n")
    print(teacher_comparison_df.round(2).to_string())

    fig, ax = plt.subplots(figsize=(9, 4.8))
    x = np.arange(len(teacher_names))
    width = 0.25
    ax.bar(x - width, list(raw_pretrained_accs.values()), width, label='Raw pretrained', color='#F18F01')
    ax.bar(x, list(linear_probe_accs.values()), width, label='Linear probe', color='#2E86AB')
    ax.bar(x + width, list(full_ft_accs.values()), width, label='Fine-tuned', color='#A23B72')
    ax.set_xticks(x)
    ax.set_xticklabels(teacher_names)
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: Teacher Standalone Performance')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{img_dir}/teacher_standalone_comparison.png', dpi=150)
    plt.show()

    class WeightedDistillationLoss(nn.Module):
        
        def __init__(self, num_teachers, alpha=0.5, temperature=3.0):
            super().__init__()
            self.alpha = alpha
            self.temperature = temperature
            self.ce_loss = nn.CrossEntropyLoss()
            self.kl_div = nn.KLDivLoss(reduction="batchmean")
            self.teacher_weights = nn.Parameter(torch.ones(num_teachers))

        def combine(self, teacher_logits_list):
            w = F.softmax(self.teacher_weights, dim=0)
            return sum(wi * tl for wi, tl in zip(w, teacher_logits_list))

        def forward(self, student_logits, teacher_logits_list, ground_truth):
            combined_teacher_logits = self.combine(teacher_logits_list)
            hard_loss = self.ce_loss(student_logits, ground_truth)
            soft_loss = self.kl_div(
                F.log_softmax(student_logits / self.temperature, dim=1),
                F.softmax(combined_teacher_logits / self.temperature, dim=1)
            ) * (self.temperature ** 2)
            return self.alpha * soft_loss + (1 - self.alpha) * hard_loss


    def train_model_with_weighted_distillation(student_model, teacher_models, train_loader, val_loader,
                                                distillation_criterion, optimizer, num_epochs=1,
                                                restore_best_checkpoint=True):
    
        best_val_loss = float('inf')
        best_state = None
        final_val_acc, final_val_loss = None, None
        for epoch in range(num_epochs):
            student_model.train()
            for images, labels in train_loader:
                images, labels = images.cuda(), labels.cuda()
                optimizer.zero_grad()
                student_outputs = student_model(images)
                with torch.no_grad():
                    teacher_logits_list = [teacher(images) for teacher in teacher_models]
                loss = distillation_criterion(student_outputs, teacher_logits_list, labels)
                loss.backward()
                optimizer.step()
            train_acc, _ = evaluate_model(student_model, train_loader, distillation_criterion.ce_loss)
            val_acc, val_loss = evaluate_model(student_model, val_loader, distillation_criterion.ce_loss)
            final_val_acc, final_val_loss = val_acc, val_loss
            print(f'Epoch [{epoch+1}/{num_epochs}] Train Acc: {train_acc:.2f}%  Val Acc: {val_acc:.2f}%')
            if restore_best_checkpoint and val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in student_model.state_dict().items()}
        if restore_best_checkpoint and best_state is not None:
            student_model.load_state_dict(best_state)
            print(f'  [best checkpoint] Restored weights from best Val Loss: {best_val_loss:.4f}')
        elif not restore_best_checkpoint:
            print(f'  [no restoration] Keeping final-epoch weights '
                  f'(Val Acc: {final_val_acc:.2f}%, Val Loss: {final_val_loss:.4f})')
        return student_model


    KD_WEIGHTING_EPOCHS = GLOBAL_NUM_EPOCHS  

    print(f"[{CURRENT_DATASET}] Building fresh (non-fine-tuned) teacher copies for the "
          f"simple-vs-weighted averaging comparison...")
    teacher_vit_kd = create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_deit_kd = create_model('deit_base_patch16_224', pretrained=True, num_classes=num_classes).cuda()
    teacher_swin_kd = create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=num_classes).cuda()

    for _t in (teacher_vit_kd, teacher_deit_kd, teacher_swin_kd):
        _t.eval()
        for _p in _t.parameters():
            _p.requires_grad = False

    teacher_models_kd = [teacher_vit_kd, teacher_deit_kd, teacher_swin_kd]

    set_all_seeds(42)
    model_simple = HybridStudentModel(num_classes, grid_size=3).cuda()
    criterion_simple = DistillationLoss(alpha=0.5, temperature=3.0)
    optimizer_simple = optim.SGD(model_simple.parameters(), lr=0.01, momentum=0.9, weight_decay=_weight_decay)
    model_simple, *_ = train_model_with_distillation(
        model_simple, teacher_models_kd, train_loader, val_loader, criterion_simple, optimizer_simple,
        num_epochs=KD_WEIGHTING_EPOCHS, restore_best_checkpoint=_restore_best_checkpoint
    )
    acc_simple, _ = evaluate_model(model_simple, test_loader, criterion_simple.ce_loss)

    del model_simple
    torch.cuda.empty_cache()
    gc.collect()

    set_all_seeds(42)
    model_weighted = HybridStudentModel(num_classes, grid_size=3).cuda()
    criterion_weighted = WeightedDistillationLoss(num_teachers=len(teacher_models_kd), alpha=0.5, temperature=3.0).cuda()
    optimizer_weighted = optim.SGD(
        list(model_weighted.parameters()) + list(criterion_weighted.parameters()), lr=0.01, momentum=0.9, weight_decay=_weight_decay
    )
    model_weighted = train_model_with_weighted_distillation(
        model_weighted, teacher_models_kd, train_loader, val_loader, criterion_weighted, optimizer_weighted,
        num_epochs=KD_WEIGHTING_EPOCHS, restore_best_checkpoint=_restore_best_checkpoint
    )
    acc_weighted, _ = evaluate_model(model_weighted, test_loader, criterion_weighted.ce_loss)

    learned_weights = F.softmax(criterion_weighted.teacher_weights.detach(), dim=0).cpu().numpy()

    del teacher_vit_kd, teacher_deit_kd, teacher_swin_kd, teacher_models_kd
    torch.cuda.empty_cache()
    gc.collect()

    kd_weighting_df = pd.DataFrame({
        'Simple averaging': {'Test Accuracy (%)': acc_simple},
        'Learned weighted averaging': {'Test Accuracy (%)': acc_weighted},
    }).T
    print(f"Table: Simple vs. learned-weighted teacher averaging in MTKD -- {CURRENT_DATASET}\n")
    print(kd_weighting_df.round(2).to_string())
    print(f"\nLearned teacher weights (softmax-normalized): "
          f"ViT={learned_weights[0]:.3f}  DeiT={learned_weights[1]:.3f}  Swin={learned_weights[2]:.3f}")

    del model_weighted
    torch.cuda.empty_cache()
    gc.collect()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(['Simple avg', 'Learned weighted'], [acc_simple, acc_weighted], color=['#2E86AB', '#A23B72'])
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{CURRENT_DATASET}: MTKD Simple vs. Weighted Teacher Averaging')
    plt.tight_layout()
    plt.savefig(f'{img_dir}/kd_weighting_comparison.png', dpi=150)
    plt.show()



    import re

    def try_extract_patient_id(filename):
        """Attempts several common histopathology dataset patient/case-ID filename
        conventions. Returns None if no pattern matches -the audit below treats that
        as 'cannot verify from filenames alone' rather than silently assuming safety."""
        patterns = [
            r'SOB_[A-Z]_[A-Z]+-(\d+-\d+)',   
            r'^(P\d+)',                        
            r'patient[_-]?(\d+)',              
            r'case[_-]?(\d+)',                 
        ]
        for pat in patterns:
            m = re.search(pat, filename, flags=re.IGNORECASE)
            if m:
                return m.group(1)
        return None


    def resolve_nested_subset_indices(subset):
        
        indices = list(subset.indices)
        base = subset.dataset
        while isinstance(base, torch.utils.data.Subset):
            indices = [base.indices[i] for i in indices]
            base = base.dataset
        return indices, base


    def audit_split_for_leakage(train_subset, val_subset, test_subset):
        train_idx, base_ds = resolve_nested_subset_indices(train_subset)
        val_idx, _ = resolve_nested_subset_indices(val_subset)
        test_idx, _ = resolve_nested_subset_indices(test_subset)

        filepaths = [s[0] for s in base_ds.samples]
        patient_ids = [try_extract_patient_id(os.path.basename(fp)) for fp in filepaths]
        n_matched = sum(1 for p in patient_ids if p is not None)

        print(f"Filenames with a recognizable patient/case ID pattern: {n_matched}/{len(filepaths)}")

        if n_matched == 0:
            print(f"\nNo patient/case ID could be parsed from filenames in the {CURRENT_DATASET} dataset folder.")
            print(f"For {CURRENT_DATASET}, this means no patient/slide metadata was recognized in the")
            print("filenames (or this dataset genuinely has no patient/slide grouping, e.g. a tile-level")
            print("texture benchmark like Kather5k/NCT100k). If that's expected for this dataset, state it")
            print("explicitly in the rebuttal rather than claiming a patient-level split it doesn't support.")
            print(f"If {CURRENT_DATASET} DOES have multiple patches per patient/slide (e.g. BreakHis),")
            print("extend try_extract_patient_id()'s regex patterns to match its actual filename")
            print("convention, then rerun this cell and verify zero overlap before reporting final numbers.")
            return None

        train_patients = set(patient_ids[i] for i in train_idx if patient_ids[i] is not None)
        val_patients = set(patient_ids[i] for i in val_idx if patient_ids[i] is not None)
        test_patients = set(patient_ids[i] for i in test_idx if patient_ids[i] is not None)

        overlap_train_test = train_patients & test_patients
        overlap_train_val = train_patients & val_patients
        overlap_val_test = val_patients & test_patients

        print(f"\nUnique patients -- train: {len(train_patients)}, val: {len(val_patients)}, test: {len(test_patients)}")
        print(f"Patient overlap train<->test: {len(overlap_train_test)}")
        print(f"Patient overlap train<->val:  {len(overlap_train_val)}")
        print(f"Patient overlap val<->test:   {len(overlap_val_test)}")

        if overlap_train_test or overlap_train_val or overlap_val_test:
            print("\nLEAKAGE DETECTED: at least one patient appears in more than one split.")
            print("ACTION NEEDED: re-split at the patient level (e.g. sklearn GroupShuffleSplit /")
            print("GroupKFold using patient_ids as the group key) before reporting final numbers.")
        else:
            print("\nNo patient overlap detected across splits (based on parsed IDs).")

        return {'train': train_patients, 'val': val_patients, 'test': test_patients}


    print(f"[{CURRENT_DATASET}] Auditing the split from Section 1 / Cell 7 (dataset_path = {dataset_path})\n")
    patient_audit_result = audit_split_for_leakage(train_dataset, val_dataset, test_dataset)


    dataset_results = {
        'comparison_results': comparison_results,
        'metrics_table_3': metrics_table,
        'efficiency_table_5': efficiency_table,
        'ablation_df': ablation_df,
        'stat_summary_df': stat_summary_df,
        'foundation_comparison_df': foundation_comparison_df,
        'kfold_summary_df': kfold_summary_df,
        'hparam_layers_df': layers_df,
        'hparam_heads_df': heads_df,
        'teacher_comparison_df': teacher_comparison_df,
        'kd_weighting_df': kd_weighting_df,
        'sota_comparison_df': sota_comparison_df,
        'patient_audit_result': patient_audit_result,
    }
    print(f"\n{'#'*80}\n# FINISHED PIPELINE FOR DATASET: {CURRENT_DATASET}\n{'#'*80}\n")
    return dataset_results


In [ ]:
all_dataset_results = {}

for _ds_name in DATASETS_TO_RUN:
    _ds_cfg = DATASET_CONFIGS[_ds_name]

    _missing = _check_dataset_paths(_ds_cfg)
    if _missing:
        print(f"\n[{_ds_name}] SKIPPED -- path(s) not found: {_missing}")
        print(f"[{_ds_name}] Update DATASET_CONFIGS['{_ds_name}'] in the dataset-registry cell "
              f"above (or set DATA_ROOT) to point at your actual data location.\n")
        all_dataset_results[_ds_name] = {"error": f"path(s) not found: {_missing}"}
        continue

    try:
        all_dataset_results[_ds_name] = run_pipeline(_ds_name, _ds_cfg)
    except Exception as e:
        print(f"\n[{_ds_name}] PIPELINE FAILED -- {type(e).__name__}: {e}")
        print(f"[{_ds_name}] Skipping to the next dataset (if any).\n")
        all_dataset_results[_ds_name] = {"error": str(e)}
    finally:
        torch.cuda.empty_cache()
        gc.collect()

print(f"\n\nDone. Ran: {list(all_dataset_results.keys())}")


In [ ]:
for _ds_name, _res in all_dataset_results.items():
    print(f"\n=== {_ds_name} ===")
    if "error" in _res:
        print(f"  FAILED: {_res['error']}")
        continue
    print(_res["comparison_results"])
